# Global Migration Patterns: A Data-Driven Story (1990-2024)

## Interactive Visualisations

**Student:** Ana Paiva  

---

## The Story We Tell

In the past three decades, over **300 million people** have moved across international borders - seeking better lives, fleeing conflict, or reuniting with family. This dashboard explores:

1. **Where do migrants go?** - The world's top destination countries
2. **Where do they come from?** - Countries with the largest diasporas
3. **Why do they move?** - Economic opportunity vs. forced displacement
4. **How has this changed?** - The dramatic shifts since 1990, especially post-2020

Immigration has become one of the most debated political issues globally. From the European migrant crisis to debates in North America and Australia, understanding migration data helps inform these crucial discussions.

---

### Data Sources

| Dataset | Source | Coverage | Purpose |
|---------|--------|----------|--------|
| International Migrant Stock | UN DESA Population Division | 1990-2024, 230+ countries | Total foreign-born population |
| Refugee Statistics | UNHCR Global Trends reports (approximate figures typed in) | Five crisis origins | Flags forced displacement |

Available at: https://population.un.org/ and https://data.worldbank.org/

---

## Section 1: Environment Setup
Install and import all required packages for interactive visualisations.

In [ ]:
# =============================================================================
# SECTION 1: INSTALL PACKAGES
# =============================================================================
# Run this cell first in Google Colab to install required packages
!pip install plotly pandas openpyxl ipywidgets wbgapi -q
print("All packages installed successfully!")

In [ ]:
# =============================================================================
# SECTION 1B: IMPORT LIBRARIES
# =============================================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')
# Configure Plotly for Google Colab
import plotly.io as pio
pio.renderers.default = 'colab'
# Optional: the notebook first tries the World Bank API for refugee data
try:
    import wbgapi as wb
    print("World Bank API available for refugee data")
except:
    print("Note: wbgapi not available, the notebook will use the built-in refugee estimates")
print("\nAll libraries imported successfully!")
print("Python version:", pd.__version__)

## Section 2: Data Upload

### Required Files
Download these Excel files from the UN Population Division, then select all 3 at once when the upload window opens. If any file is missing or wrong, the notebook tells you which one and asks again:
1. `undesa_pd_2024_ims_stock_by_sex_and_destination.xlsx`
2. `undesa_pd_2024_ims_stock_by_sex_and_origin.xlsx`
3. `undesa_pd_2024_ims_stock_by_sex_destination_and_origin.xlsx`

In [ ]:
# =============================================================================
# SECTION 2: UPLOAD DATA FILES
# =============================================================================
import os
import re
from google.colab import files

# The three UN Excel files the notebook needs (keep these exact names)
REQUIRED_FILES = [
    'undesa_pd_2024_ims_stock_by_sex_and_destination.xlsx',
    'undesa_pd_2024_ims_stock_by_sex_and_origin.xlsx',
    'undesa_pd_2024_ims_stock_by_sex_destination_and_origin.xlsx'
]

def missing_files():
    """Return the required UN files that are not in the working folder."""
    return [name for name in REQUIRED_FILES if not os.path.exists(name)]

print("=" * 60)
print("STEP 1: Upload your UN Migration Excel files")
print("=" * 60)

# Ask for the files (up to 3 attempts) until all of them are present
for attempt in range(3):
    missing = missing_files()
    if not missing:
        break

    print("\nSelect ALL 3 files at once in the upload window:")
    for name in REQUIRED_FILES:
        status = "[MISSING]" if name in missing else "[OK]     "
        print(f"  {status} {name}")
    print()

    uploaded = files.upload()

    for name in list(uploaded.keys()):
        # Colab renames repeated uploads, e.g. 'file (1).xlsx'
        base_name = re.sub(r" \(\d+\)(?=\.xlsx$)", "", name)
        if base_name != name and base_name in REQUIRED_FILES:
            os.replace(name, base_name)
        elif name not in REQUIRED_FILES:
            print(f"  Ignored (not one of the 3 UN files): {name}")

missing = missing_files()
if missing:
    print("\nStill missing:")
    for name in missing:
        print(f"  - {name}")
    print("\nRun this cell again and select the missing file(s).")
else:
    print("\nAll 3 UN files are ready!")
    for name in REQUIRED_FILES:
        print(f"  - {name}")

## Section 3: Design Principles and Colour Scheme

### Tufte's Visualisation Principles Applied:
1. **High data-ink ratio** - Minimal non-data elements
2. **No chartjunk** - Clean, uncluttered design
3. **Clear labelling** - Informative titles and legends
4. **Appropriate colour** - Avoiding blue (water) and green (vegetation) for data values

### Colour Scheme Rationale:
- **Orange scale** for sequential data (population counts)
- **Purple scale** for alternative sequential data
- **Purple-Orange diverging** for net migration (colourblind-safe)

In [ ]:
# =============================================================================
# SECTION 3: COLOUR SCHEMES AND STYLING
# =============================================================================
# Following cartographic conventions: avoid blue (water) and green (vegetation)
# Sequential scale: Oranges - for immigrant/emigrant counts
COLOUR_SCALE_ORANGE = [
    [0.0, '#fff5eb'],   # Very light
    [0.2, '#fee6ce'],   # Light orange
    [0.4, '#fdae6b'],   # Medium orange
    [0.6, '#f16913'],   # Orange
    [0.8, '#d94801'],   # Dark orange
    [1.0, '#8c2d04']    # Very dark (brown)
]
# Alternative sequential: Purples
COLOUR_SCALE_PURPLE = [
    [0.0, '#fcfbfd'],
    [0.2, '#dadaeb'],
    [0.4, '#9e9ac8'],
    [0.6, '#756bb1'],
    [0.8, '#54278f'],
    [1.0, '#3f007d']
]
# Diverging scale: Purple-Orange (colourblind safe)
# Purple = negative (net emigration), Orange = positive (net immigration)
COLOUR_SCALE_DIVERGING = [
    [0.0, '#7b3294'],   # Dark purple (loss)
    [0.25, '#c2a5cf'],  # Light purple
    [0.5, '#f7f7f7'],   # White (neutral)
    [0.75, '#fdae61'],  # Light orange
    [1.0, '#e66101']    # Dark orange (gain)
]
# Standardised plot layout (Tufte-inspired minimalism)
PLOT_LAYOUT = dict(
    font=dict(family='Arial, sans-serif', size=12, color='#333333'),
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(l=60, r=60, t=80, b=60),
    title_x=0.5  # Center titles
)
# Geographic layout for maps
GEO_LAYOUT = dict(
    showframe=False,
    showcoastlines=True,
    coastlinecolor='#888888',
    showland=True,
    landcolor='#f5f5f5',
    showocean=True,
    oceancolor='#d4e6f1',  # Light blue for water (appropriate)
    projection_type='natural earth',
    showlakes=False
)
print("Colour schemes and layouts defined!")
print("Using: Orange (sequential), Purple (alternative), Purple-Orange (diverging)")

# Census years available in the UN files (used by every time-series chart)
YEARS = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024]


## Section 4: Data Loading Functions
Functions to load, clean, and prepare the UN migration datasets.

In [ ]:
# =============================================================================
# SECTION 4: DATA LOADING AND CLEANING FUNCTIONS
# =============================================================================
# File names (standard UN naming convention)
FILE_DESTINATION = 'undesa_pd_2024_ims_stock_by_sex_and_destination.xlsx'
FILE_ORIGIN = 'undesa_pd_2024_ims_stock_by_sex_and_origin.xlsx'
FILE_MATRIX = 'undesa_pd_2024_ims_stock_by_sex_destination_and_origin.xlsx'
def identify_countries(df, name_col):
    """
    Filter to individual countries only, excluding regional aggregates.

    The UN data includes both countries AND regional groupings (e.g., 'AFRICA',
    'More developed regions'). This function identifies and keeps only countries.

    Parameters:
    -----------
    df : DataFrame - Input data
    name_col : str - Column containing country/region names

    Returns:
    --------
    DataFrame with only country-level rows
    """
    # List of regional aggregate names to exclude
    exclude_names = [
        'World', 'WORLD', 'AFRICA', 'EUROPE', 'ASIA', 'AMERICA', 'OCEANIA',
        'More developed regions', 'Less developed regions',
        'Least developed countries', 'Eastern Africa', 'Middle Africa',
        'Northern Africa', 'Southern Africa', 'Western Africa',
        'Central Asia', 'Eastern Asia', 'South-Eastern Asia', 'Southern Asia',
        'Western Asia', 'Eastern Europe', 'Northern Europe', 'Southern Europe',
        'Western Europe', 'Caribbean', 'Central America', 'South America',
        'Melanesia', 'Micronesia', 'Polynesia', 'Channel Islands',
        'NORTHERN AMERICA', 'Northern America', 'Australia/New Zealand',
        'Sub-Saharan Africa', 'Northern Africa and Western Asia',
        'Central and Southern Asia', 'Eastern and South-Eastern Asia',
        'Latin America and the Caribbean', 'Europe and Northern America',
        'Oceania (excluding Australia and New Zealand)',
        'Land-locked Developing Countries', 'Small Island Developing States'
    ]

    # Patterns that indicate regional aggregates
    exclude_patterns = ['income', 'developed', 'countries', 'regions', 'SDG', 'LLDC', 'SIDS']

    def is_country(name):
        """Return True if the name is a country, not a region or income group."""
        name_str = str(name).strip()

        # Check explicit exclusion list
        for region in exclude_names:
            if name_str == region:
                return False

        # Check patterns
        for pattern in exclude_patterns:
            if pattern.lower() in name_str.lower():
                return False

        # Exclude ALL CAPS names (regions tend to be capitalised)
        if name_str.isupper() and len(name_str) > 5:
            return False

        return True

    return df[df[name_col].apply(is_country)].copy()
def load_destination_data(filepath):
    """
    Load UN data on immigrants by destination country.

    This shows the number of foreign-born people LIVING IN each country.
    Example: How many immigrants live in Australia?

    Returns: DataFrame with columns [country, location_code, immigrants_1990, ..., immigrants_2024]
    """
    print("Loading destination (immigrants) data...")

    # Read Excel, skip header rows
    df = pd.read_excel(filepath, sheet_name='Table 1', skiprows=10)

    # Select relevant columns
    year_cols = YEARS
    cols = ['Region, development group, country or area', 'Location code'] + year_cols
    df = df[cols].copy()
    df.columns = ['country', 'location_code'] + [f'immigrants_{y}' for y in year_cols]

    # Filter to countries only
    df = identify_countries(df, 'country')

    # Clean country names (remove asterisks from footnotes)
    df['country'] = df['country'].str.replace('*', '', regex=False).str.strip()

    # Convert to numeric
    for col in df.columns[2:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f"  Loaded {len(df)} countries")
    return df
def load_origin_data(filepath):
    """
    Load UN data on emigrants by origin country.

    This shows the number of people BORN IN each country who live abroad.
    Example: How many Brazilians live outside Brazil?

    Returns: DataFrame with columns [country, location_code, emigrants_1990, ..., emigrants_2024]
    """
    print("Loading origin (emigrants) data...")

    df = pd.read_excel(filepath, sheet_name='Table 1', skiprows=10)
    year_cols = YEARS
    cols = ['Region, development group, country or area', 'Location code'] + year_cols
    df = df[cols].copy()
    df.columns = ['country', 'location_code'] + [f'emigrants_{y}' for y in year_cols]

    df = identify_countries(df, 'country')
    df['country'] = df['country'].str.replace('*', '', regex=False).str.strip()

    for col in df.columns[2:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f"  Loaded {len(df)} countries")
    return df
def load_corridor_data(filepath):
    """
    Load UN data on migration corridors (origin-destination pairs).

    This shows specific flows: e.g., how many Mexicans live in the USA?

    Returns: DataFrame with columns [destination, origin, migrants_1990, ..., migrants_2024]
    """
    print("Loading corridor (origin-destination) data...")

    df = pd.read_excel(filepath, sheet_name='Table 1', skiprows=10)
    year_cols = YEARS

    df = df[[
        'Region, development group, country or area of destination',
        'Location code of destination',
        'Region, development group, country or area of origin',
        'Location code of origin'
    ] + year_cols].copy()

    df.columns = ['destination', 'dest_code', 'origin', 'orig_code'] + [f'migrants_{y}' for y in year_cols]

    # Filter using location codes (countries have codes 4-899)
    df = df[(df['dest_code'] >= 4) & (df['dest_code'] <= 899) &
            (df['orig_code'] >= 4) & (df['orig_code'] <= 899)].copy()

    # Additional name-based filtering for safety.
    # World and continent names are matched exactly: a substring match would also
    # drop real countries such as 'United States of America' and 'South Africa'.
    exclude_exact = ['World', 'WORLD', 'Africa', 'Europe', 'Asia', 'America', 'Oceania']
    exclude_kw = ['regions', 'countries', 'income', 'OTHER']
    for col_name in ['destination', 'origin']:
        df = df[~df[col_name].isin(exclude_exact)]
        for kw in exclude_kw:
            df = df[~df[col_name].str.contains(kw, case=False, na=False)]

    df['destination'] = df['destination'].str.replace('*', '', regex=False).str.strip()
    df['origin'] = df['origin'].str.replace('*', '', regex=False).str.strip()

    for col in [c for c in df.columns if 'migrants' in c]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f"  Loaded {len(df)} country-to-country corridors")
    return df
def create_combined_dataset(df_dest, df_orig):
    """
    Merge destination and origin data to calculate net migration.

    Net migration = Immigrants - Emigrants
    Positive: More people arriving than leaving (net receiver)
    Negative: More people leaving than arriving (net sender)
    """
    print("Creating combined dataset...")

    df = pd.merge(
        df_dest,
        df_orig[['country'] + [c for c in df_orig.columns if 'emigrants' in c]],
        on='country', how='inner'
    )

    # Calculate net migration for each year
    for year in YEARS:
        df[f'net_migration_{year}'] = df[f'immigrants_{year}'] - df[f'emigrants_{year}']

    # Calculate growth rates (1990 to 2024)
    df['immigrant_growth_pct'] = ((df['immigrants_2024'] - df['immigrants_1990']) /
                                   df['immigrants_1990'] * 100).round(1)
    df['emigrant_growth_pct'] = ((df['emigrants_2024'] - df['emigrants_1990']) /
                                  df['emigrants_1990'] * 100).round(1)

    # Calculate post-COVID growth (2020 to 2024)
    df['immigrant_growth_post_covid'] = ((df['immigrants_2024'] - df['immigrants_2020']) /
                                          df['immigrants_2020'] * 100).round(1)

    print(f"  Combined: {len(df)} countries with all metrics")
    return df
print("Data loading functions defined!")

## Section 5: Load and Verify Data

In [ ]:
# =============================================================================
# SECTION 5: LOAD ALL DATASETS
# =============================================================================
print("=" * 70)
print("LOADING UN MIGRATION DATA")
print("=" * 70)
# Load the three UN datasets
df_destination = load_destination_data(FILE_DESTINATION)
df_origin = load_origin_data(FILE_ORIGIN)
df_corridors = load_corridor_data(FILE_MATRIX)
# Create combined dataset with net migration
df_combined = create_combined_dataset(df_destination, df_origin)
# Verification
print("\n" + "=" * 70)
print("DATA VERIFICATION")
print("=" * 70)
# Check key countries are present
key_countries = ['Australia', 'Brazil', 'United States of America', 'Germany',
                 'India', 'China', 'Syrian Arab Republic', 'Ukraine']
for country in key_countries:
    present = country in df_combined['country'].values
    symbol = "OK" if present else "MISSING"
    print(f"  [{symbol}] {country}")
print(f"\nTotal countries: {len(df_combined)}")
print(f"Total corridors: {len(df_corridors)}")
print(f"Years covered: {', '.join(map(str, YEARS))}")

In [ ]:
# =============================================================================
# SECTION 5B: DATA CHECKS
# =============================================================================
# Stop early, with a clear message, if the loaded data looks wrong.
aggregate_names = {'World', 'WORLD', 'Africa', 'Europe', 'Asia', 'America', 'Oceania'}
aggregate_keywords = ['regions', 'countries', 'income', 'OTHER']
results = []

def check(label, passed):
    """Record one data check and print it as OK or FAIL."""
    results.append((label, bool(passed)))
    print(f"  [{'OK' if passed else 'FAIL'}] {label}")

print("=" * 70)
print("DATA CHECKS")
print("=" * 70)
needed = [f'{m}_{y}' for m in ('immigrants', 'emigrants', 'net_migration') for y in YEARS]
check("every immigrant, emigrant and net migration column exists for all census years",
      all(c in df_combined.columns for c in needed))
check("each country appears once", df_combined['country'].is_unique)
check("no regional or income-group aggregates among the countries",
      not df_combined['country'].isin(aggregate_names).any() and
      not any(df_combined['country'].str.contains(k, case=False, na=False).any() for k in aggregate_keywords))
check("no negative immigrant or emigrant counts",
      (df_combined[[c for c in needed if not c.startswith('net_')]] < 0).sum().sum() == 0)
world_2024 = df_combined['immigrants_2024'].sum()
check(f"world migrant stock in 2024 is plausible ({world_2024/1e6:.0f} million, expected about 300)",
      290e6 <= world_2024 <= 320e6)
gap = abs(df_combined['emigrants_2024'].sum() - world_2024) / world_2024
check(f"total emigrants and total immigrants agree within 10% (gap {gap*100:.1f}%)", gap < 0.10)
check("no corridor has the same origin and destination",
      (df_corridors['origin'] != df_corridors['destination']).all())
failed = [label for label, passed in results if not passed]
assert not failed, "Data checks failed: " + "; ".join(failed)
print(f"\nAll {len(results)} checks passed.")

## Section 6: Exploratory Data Analysis (EDA)

Before creating visualisations, we must understand our data:
- Data types and structure
- Missing values
- Distribution of key variables
- Summary statistics

In [ ]:
# =============================================================================
# SECTION 6A: DATA STRUCTURE AND TYPES
# =============================================================================
print("=" * 70)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 70)
print("\n--- Combined Dataset Structure ---")
print(f"Shape: {df_combined.shape[0]} rows x {df_combined.shape[1]} columns")
print("\nColumn names and data types:")
print(df_combined.dtypes)
print("\n--- Sample Data (first 5 rows) ---")
display(df_combined[['country', 'immigrants_2024', 'emigrants_2024',
                     'net_migration_2024', 'immigrant_growth_pct']].head())

In [ ]:
# =============================================================================
# SECTION 6B: MISSING VALUES ANALYSIS
# =============================================================================
print("--- Missing Values Analysis ---\n")
# Count missing values per column
missing = df_combined.isnull().sum()
missing_pct = (missing / len(df_combined) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0]
if len(missing_df) > 0:
    print("Columns with missing values:")
    display(missing_df)
else:
    print("No missing values found in the dataset!")
# Check for zeros (might indicate data issues)
print("\n--- Zero Value Check (2024 data) ---")
zeros_imm = (df_combined['immigrants_2024'] == 0).sum()
zeros_emi = (df_combined['emigrants_2024'] == 0).sum()
print(f"Countries with 0 immigrants: {zeros_imm}")
print(f"Countries with 0 emigrants: {zeros_emi}")

In [ ]:
# =============================================================================
# SECTION 6C: SUMMARY STATISTICS
# =============================================================================
print("--- Summary Statistics (2024) ---\n")
stats_cols = ['immigrants_2024', 'emigrants_2024', 'net_migration_2024',
              'immigrant_growth_pct', 'emigrant_growth_pct']
summary = df_combined[stats_cols].describe().round(0)
summary.columns = ['Immigrants', 'Emigrants', 'Net Migration',
                   'Imm. Growth %', 'Emi. Growth %']
display(summary)
print("\n--- Key Insights ---")
total_migrants = df_combined['immigrants_2024'].sum()
mean_migrants = df_combined['immigrants_2024'].mean()
median_migrants = df_combined['immigrants_2024'].median()
print(f"Total migrants worldwide (2024): {total_migrants:,.0f}")
print(f"Mean per country: {mean_migrants:,.0f}")
print(f"Median per country: {median_migrants:,.0f}")
print(f"\nNote: Mean >> Median indicates highly skewed distribution")
print("(A few countries host a large share of migrants: the 10 largest destinations host about half)")

In [ ]:
# =============================================================================
# SECTION 6D: DISTRIBUTION ANALYSIS
# =============================================================================
# Create distribution plots
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Immigrants 2024 (Log Scale)',
                                    'Emigrants 2024 (Log Scale)',
                                    'Net Migration 2024',
                                    'Immigrant Growth % (1990-2024)'))
# Plot 1: Immigrants distribution (log scale due to skewness)
fig.add_trace(
    go.Histogram(x=np.log10(df_combined['immigrants_2024'].clip(lower=1)),
                 nbinsx=30, marker_color='#fdae6b', name='Immigrants'),
    row=1, col=1)
# Plot 2: Emigrants distribution (log scale)
fig.add_trace(
    go.Histogram(x=np.log10(df_combined['emigrants_2024'].clip(lower=1)),
                 nbinsx=30, marker_color='#9e9ac8', name='Emigrants'),
    row=1, col=2)
# Plot 3: Net migration distribution
fig.add_trace(
    go.Histogram(x=df_combined['net_migration_2024'],
                 nbinsx=40, marker_color='#66c2a4', name='Net Migration'),
    row=2, col=1)
# Plot 4: Growth rate distribution (filtered for outliers)
growth_filtered = df_combined['immigrant_growth_pct'].clip(-100, 1000)
fig.add_trace(
    go.Histogram(x=growth_filtered,
                 nbinsx=40, marker_color='#fc8d62', name='Growth %'),
    row=2, col=2)
fig.update_layout(
    title='<b>Distribution of Key Variables</b>',
    showlegend=False,
    height=500,
    **PLOT_LAYOUT
)
fig.update_xaxes(title_text='Log10(Immigrants)', row=1, col=1)
fig.update_xaxes(title_text='Log10(Emigrants)', row=1, col=2)
fig.update_xaxes(title_text='Net Migration', row=2, col=1)
fig.update_xaxes(title_text='Growth %', row=2, col=2)
fig.show()
print("\nKey observation: Data is highly right-skewed - a few countries dominate.")

In [ ]:
# =============================================================================
# SECTION 6E: TOP COUNTRIES OVERVIEW
# =============================================================================
print("=" * 70)
print("TOP 10 COUNTRIES BY KEY METRICS (2024)")
print("=" * 70)
# Top destinations
print("\n--- Top 10 Destination Countries (Most Immigrants) ---")
top_dest = df_combined.nlargest(10, 'immigrants_2024')[
    ['country', 'immigrants_2024', 'immigrants_1990', 'immigrant_growth_pct']
].copy()
top_dest['immigrants_2024'] = top_dest['immigrants_2024'].apply(lambda x: f"{x:,.0f}")
top_dest['immigrants_1990'] = top_dest['immigrants_1990'].apply(lambda x: f"{x:,.0f}")
top_dest['immigrant_growth_pct'] = top_dest['immigrant_growth_pct'].apply(lambda x: f"{x:+.0f}%")
top_dest.columns = ['Country', 'Immigrants 2024', 'Immigrants 1990', 'Growth']
display(top_dest.reset_index(drop=True))
# Top origins
print("\n--- Top 10 Origin Countries (Most Emigrants) ---")
top_orig = df_combined.nlargest(10, 'emigrants_2024')[
    ['country', 'emigrants_2024', 'emigrants_1990', 'emigrant_growth_pct']
].copy()
top_orig['emigrants_2024'] = top_orig['emigrants_2024'].apply(lambda x: f"{x:,.0f}")
top_orig['emigrants_1990'] = top_orig['emigrants_1990'].apply(lambda x: f"{x:,.0f}")
top_orig['emigrant_growth_pct'] = top_orig['emigrant_growth_pct'].apply(lambda x: f"{x:+.0f}%")
top_orig.columns = ['Country', 'Emigrants 2024', 'Emigrants 1990', 'Growth']
display(top_orig.reset_index(drop=True))
# Biggest net receivers and senders
print("\n--- Biggest Net Receivers (Gain population through migration) ---")
net_receivers = df_combined.nlargest(5, 'net_migration_2024')[['country', 'net_migration_2024']]
for _, row in net_receivers.iterrows():
    print(f"  {row['country']}: +{row['net_migration_2024']:,.0f}")
print("\n--- Biggest Net Senders (Lose population through migration) ---")
net_senders = df_combined.nsmallest(5, 'net_migration_2024')[['country', 'net_migration_2024']]
for _, row in net_senders.iterrows():
    print(f"  {row['country']}: {row['net_migration_2024']:,.0f}")

In [ ]:
# =============================================================================
# SECTION 6F: TEMPORAL TRENDS
# =============================================================================
print("=" * 70)
print("GLOBAL TRENDS OVER TIME")
print("=" * 70)
years = YEARS
totals = [df_combined[f'immigrants_{y}'].sum() for y in years]
# Calculate period growth rates
print("\n--- Global Migrant Stock ---")
for i, year in enumerate(years):
    if i > 0:
        growth = (totals[i] - totals[i-1]) / totals[i-1] * 100
        print(f"  {year}: {totals[i]/1e6:,.0f} million (+{growth:.1f}% from {years[i-1]})")
    else:
        print(f"  {year}: {totals[i]/1e6:,.0f} million (baseline)")
# Overall growth
overall_growth = (totals[-1] - totals[0]) / totals[0] * 100
print(f"\n  Overall growth 1990-2024: +{overall_growth:.0f}%")
# Create trend chart
fig_trend = go.Figure()
fig_trend.add_trace(go.Scatter(
    x=years, y=[t/1e6 for t in totals],
    mode='lines+markers+text',
    text=[f'{t/1e6:.0f}M' for t in totals],
    textposition='top center',
    line=dict(color='#d94801', width=3),
    marker=dict(size=10)
))
fig_trend.update_layout(
    title='<b>Global International Migrants (1990-2024)</b><br><sup>Total foreign-born population worldwide</sup>',
    xaxis_title='Year',
    yaxis_title='Millions of People',
    height=400,
    **PLOT_LAYOUT
)
fig_trend.show()
print("\nKey insight: Migration has doubled since 1990, with acceleration post-2015.")

## Section 7: UNHCR Refugee Data Integration

### Distinguishing Types of Migration

The UN migrant stock data counts ALL foreign-born people, but does not distinguish between:
- **Voluntary migration**: Economic migrants, students, family reunification
- **Forced displacement**: Refugees, asylum seekers fleeing conflict or persecution

By integrating UNHCR refugee data, we can tell a more complete story about **why** people move.

### Data Source
The notebook first tries the World Bank API (indicator SM.POP.REFG). That indicator is no longer published, so the call fails and the notebook uses approximate figures typed in from UNHCR Global Trends reports for five crisis origins (Syria, Ukraine, Afghanistan, Venezuela and South Sudan). These estimates are only used to flag crisis origins. They are not used in any headline number.

In [ ]:
# =============================================================================
# SECTION 7: LOAD UNHCR REFUGEE DATA
# =============================================================================
# Using World Bank API to fetch UNHCR refugee statistics
print("=" * 70)
print("LOADING UNHCR REFUGEE DATA")
print("=" * 70)
try:
    import wbgapi as wb
    # Fetch refugee data by host country
    # SM.POP.REFG = Refugees by country of asylum
    print("\nFetching refugee data from World Bank API...")
    refugee_data = wb.data.DataFrame(
        'SM.POP.REFG',
        # Refugees by country of asylum
        time=range(1990, 2024),
        labels=True
    )
    # Reshape for easier use
    refugee_data = refugee_data.reset_index()
    refugee_data = refugee_data.melt(id_vars=['economy'],
                                     var_name='year',
                                     value_name='refugees')
    refugee_data['year'] = refugee_data['year'].str.replace('YR', '').astype(int)
    # Get country names
    countries_info = wb.economy.info()
    print(f"  Loaded refugee data for {refugee_data['economy'].nunique()} countries")
    print(f"  Years: {refugee_data['year'].min()} to {refugee_data['year'].max()}")
    REFUGEE_DATA_AVAILABLE = True
except Exception as e:
    print(f"  Could not load World Bank data: {e}")
    print("  Using alternative approach with manually created refugee estimates...")
    REFUGEE_DATA_AVAILABLE = False
    # Approximate figures typed in from UNHCR Global Trends reports (not fetched, not verified here).
    # They are only used to flag crisis origins and are not used in any headline number.
    refugee_estimates = {
        'Syrian Arab Republic': {'origin_2024': 6100000, 'origin_2015': 4900000, 'origin_1990': 21000},
        'Ukraine': {'origin_2024': 6300000, 'origin_2015': 6400, 'origin_1990': 3800},
        'Afghanistan': {'origin_2024': 6400000, 'origin_2015': 2700000, 'origin_1990': 6300000},
        'Venezuela (Bolivarian Republic of)': {'origin_2024': 6100000, 'origin_2015': 700, 'origin_1990': 200},
        'South Sudan': {'origin_2024': 2300000, 'origin_2015': 780000, 'origin_1990': 0},
        'Myanmar': {'origin_2024': 1300000, 'origin_2015': 451000, 'origin_1990': 78000},
        'Türkiye': {'host_2024': 3200000, 'host_2015': 2500000, 'host_1990': 23000},
        'Germany': {'host_2024': 2600000, 'host_2015': 316000, 'host_1990': 438000},
        'Pakistan': {'host_2024': 2100000, 'host_2015': 1500000, 'host_1990': 3250000},
        'Colombia': {'host_2024': 2900000, 'host_2015': 300, 'host_1990': 200},
        'Uganda': {'host_2024': 1700000, 'host_2015': 477000, 'host_1990': 142000}
    }
    df_refugees = pd.DataFrame([
        {'country': k, **v} for k, v in refugee_estimates.items()
    ])
    print("  Created refugee estimates dataset from UNHCR reports")
    display(df_refugees)

In [ ]:
# =============================================================================
# SECTION 7B: INTEGRATE REFUGEE DATA WITH MIGRATION DATA
# =============================================================================
print("=" * 70)
print("REFUGEE DATA INTEGRATION")
print("=" * 70)
# Crisis classification used by the charts below (independent of where the refugee data comes from)
CRISIS_ORIGINS = ['Syrian Arab Republic', 'Ukraine', 'Afghanistan',
                  'Venezuela (Bolivarian Republic of)', 'South Sudan']
df_combined['is_crisis_origin'] = df_combined['country'].isin(CRISIS_ORIGINS)

if REFUGEE_DATA_AVAILABLE:
    # Get 2022 data (latest complete year in World Bank)
    refugees_2022 = refugee_data[refugee_data['year'] == 2022].copy()
    refugees_2022 = refugees_2022[['economy', 'refugees']].rename(
        columns={'economy': 'country', 'refugees': 'refugees_hosted'}
    )

    # Merge with combined data
    df_combined = df_combined.merge(refugees_2022, on='country', how='left')
    df_combined['refugees_hosted'] = df_combined['refugees_hosted'].fillna(0)

    # Calculate refugee percentage of total immigrants
    df_combined['refugee_pct'] = (
        df_combined['refugees_hosted'] / df_combined['immigrants_2024'] * 100
    ).round(1)

    print("Merged refugee data with migration data")

else:
    # Use the manual estimates - add key refugee-related columns
    # For countries with high forced displacement
    crisis_countries = {
        'Syrian Arab Republic': 6100000,
        'Ukraine': 6300000,
        'Afghanistan': 6400000,
        'Venezuela (Bolivarian Republic of)': 6100000,
        'South Sudan': 2300000
    }

    df_combined['crisis_emigrants'] = df_combined['country'].map(crisis_countries).fillna(0)

    # Calculate what percentage of emigrants are refugees (for crisis countries)
    df_combined['forced_displacement_pct'] = np.where(
        df_combined['emigrants_2024'] > 0,
        (df_combined['crisis_emigrants'] / df_combined['emigrants_2024'] * 100).round(1),
        0
    )

    print("Added crisis country indicators")
print("\n--- Countries with Significant Forced Displacement ---")
crisis_display = df_combined[df_combined['is_crisis_origin'] == True][
    ['country', 'emigrants_2024', 'emigrants_1990', 'emigrant_growth_pct']
].copy()
crisis_display.columns = ['Country', 'Emigrants 2024', 'Emigrants 1990', 'Growth %']
display(crisis_display)
print("\nNote: These countries show dramatic emigration growth driven by conflict.")

In [ ]:
# =============================================================================
# SECTION 7C: VISUALISE FORCED VS VOLUNTARY MIGRATION
# =============================================================================
# Create comparison chart: Crisis countries vs economic migration countries
print("--- Comparing Crisis Countries vs Economic Migration ---\n")
# Define categories
crisis_countries = ['Syrian Arab Republic', 'Ukraine', 'Afghanistan', 'Venezuela (Bolivarian Republic of)', 'South Sudan']
economic_origins = ['India', 'Mexico', 'China', 'Philippines', 'Bangladesh']
# Get data
crisis_data = df_combined[df_combined['country'].isin(crisis_countries)].copy()
economic_data = df_combined[df_combined['country'].isin(economic_origins)].copy()
# Create comparison bar chart
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Crisis-Driven Emigration', 'Economic Migration'))
# Crisis countries
fig.add_trace(
    go.Bar(
        y=crisis_data.sort_values('emigrants_2024')['country'],
        x=crisis_data.sort_values('emigrants_2024')['emigrants_2024'] / 1e6,
        orientation='h',
        marker_color='#d62728',  # Red for crisis
        name='Crisis',
        hovertemplate='%{y}<br>%{x:.1f}M emigrants<extra></extra>'
    ),
    row=1, col=1
)
# Economic migration countries
fig.add_trace(
    go.Bar(
        y=economic_data.sort_values('emigrants_2024')['country'],
        x=economic_data.sort_values('emigrants_2024')['emigrants_2024'] / 1e6,
        orientation='h',
        marker_color='#2ca02c',  # Green for economic
        name='Economic',
        hovertemplate='%{y}<br>%{x:.1f}M emigrants<extra></extra>'
    ),
    row=1, col=2
)
fig.update_layout(
    title='<b>Types of Migration: Crisis vs Economic Origins</b><br><sup>Emigrants by country (millions), 2024</sup>',
    showlegend=False,
    height=400,
    **PLOT_LAYOUT
)
fig.update_xaxes(title_text='Emigrants (millions)', row=1, col=1)
fig.update_xaxes(title_text='Emigrants (millions)', row=1, col=2)
fig.show()
print("Key insight: Both crisis and economic factors drive migration,")
print("but the GROWTH RATE differs dramatically for crisis countries (Syria: +2,914%)")

## Section 8: Choropleth Map Functions

Three main choropleth maps plus animated version:
1. **Immigrants by Destination** - Where migrants live
2. **Emigrants by Origin** - Where migrants come from
3. **Net Migration** - Population gain/loss through migration
4. **Animated** - Changes over time (1990-2024)

In [ ]:
# =============================================================================
# SECTION 8: CHOROPLETH MAP FUNCTIONS
# =============================================================================
def create_choropleth_immigrants(df, year=2024):
    """
    Choropleth Map 1: Immigrants by destination country.
    Shows the number of foreign-born people living in each country.

    Parameters:
        df: DataFrame with migration data
        year: Year to display (1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024)

    Returns:
        Plotly figure object
    """
    col = f'immigrants_{year}'

    fig = px.choropleth(
        df,
        locations='country',
        locationmode='country names',
        color=col,
        hover_name='country',
        hover_data={col: ':,.0f', 'country': False},
        color_continuous_scale=COLOUR_SCALE_ORANGE,
        labels={col: 'Immigrants'},
        title=f'<b>Map 1: Immigrants by Destination Country ({year})</b><br><sup>Number of foreign-born residents</sup>'
    )

    fig.update_layout(**PLOT_LAYOUT, geo=GEO_LAYOUT,
                      coloraxis_colorbar=dict(title='Immigrants', tickformat=',.0s'))
    return fig
def create_choropleth_emigrants(df, year=2024):
    """
    Choropleth Map 2: Emigrants by origin country.
    Shows the number of people born in each country who live abroad (diaspora).

    Parameters:
        df: DataFrame with migration data
        year: Year to display

    Returns:
        Plotly figure object
    """
    col = f'emigrants_{year}'

    fig = px.choropleth(
        df,
        locations='country',
        locationmode='country names',
        color=col,
        hover_name='country',
        hover_data={col: ':,.0f', 'country': False},
        color_continuous_scale=COLOUR_SCALE_PURPLE,
        labels={col: 'Emigrants'},
        title=f'<b>Map 2: Emigrants by Origin Country ({year})</b><br><sup>Citizens living abroad</sup>'
    )

    fig.update_layout(**PLOT_LAYOUT, geo=GEO_LAYOUT,
                      coloraxis_colorbar=dict(title='Emigrants', tickformat=',.0s'))
    return fig
def create_choropleth_net_migration(df, year=2024):
    """
    Choropleth Map 3: Net migration by country.
    Net = Immigrants - Emigrants
    Purple: Net loss (more leaving than arriving)
    Orange: Net gain (more arriving than leaving)

    Parameters:
        df: DataFrame with migration data
        year: Year to display

    Returns:
        Plotly figure object
    """
    col = f'net_migration_{year}'
    max_abs = max(abs(df[col].min()), abs(df[col].max()))

    fig = px.choropleth(
        df,
        locations='country',
        locationmode='country names',
        color=col,
        hover_name='country',
        hover_data={col: ':+,.0f', 'country': False},
        color_continuous_scale=COLOUR_SCALE_DIVERGING,
        range_color=[-max_abs, max_abs],
        labels={col: 'Net Migration'},
        title=f'<b>Map 3: Net Migration ({year})</b><br><sup>Purple = net loss, Orange = net gain</sup>'
    )

    fig.update_layout(**PLOT_LAYOUT, geo=GEO_LAYOUT,
                      coloraxis_colorbar=dict(title='Net', tickformat='+,.0s'))
    return fig
def create_animated_choropleth(df, metric='immigrants'):
    """
    Animated choropleth showing changes from 1990 to 2024.
    Press Play button to watch migration patterns evolve.

    Parameters:
        df: DataFrame with migration data
        metric: 'immigrants', 'emigrants', or 'net_migration'

    Returns:
        Plotly figure with animation
    """
    years = YEARS

    # Reshape to long format for animation
    dfs = []
    for year in years:
        temp = df[['country', f'{metric}_{year}']].copy()
        temp['year'] = year
        temp.columns = ['country', 'value', 'year']
        dfs.append(temp)
    df_long = pd.concat(dfs, ignore_index=True)

    titles = {
        'immigrants': 'Immigrants by Destination',
        'emigrants': 'Emigrants by Origin',
        'net_migration': 'Net Migration'
    }

    if metric == 'net_migration':
        cscale = COLOUR_SCALE_DIVERGING
        max_abs = df_long['value'].abs().max()
        rng = [-max_abs, max_abs]
    else:
        cscale = COLOUR_SCALE_ORANGE if metric == 'immigrants' else COLOUR_SCALE_PURPLE
        rng = [0, df_long['value'].max()]

    fig = px.choropleth(
        df_long,
        locations='country',
        locationmode='country names',
        color='value',
        animation_frame='year',
        color_continuous_scale=cscale,
        range_color=rng,
        title=f'<b>Animated: {titles[metric]} (1990-2024)</b><br><sup>Press Play to animate</sup>'
    )

    fig.update_layout(**PLOT_LAYOUT, geo=GEO_LAYOUT,
                      coloraxis_colorbar=dict(tickformat=',.0s'),
                      sliders=[dict(currentvalue=dict(prefix='Year: '))])

    # Slow down animation for readability
    fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200
    fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 600

    return fig
print("Choropleth functions defined!")

## Section 9: Choropleth Map 1 - Immigrants by Destination
Where do international migrants live? This map shows the foreign-born population in each country.

In [ ]:
# =============================================================================
# SECTION 9: CHOROPLETH MAP 1 - IMMIGRANTS BY DESTINATION
# =============================================================================
fig_immigrants = create_choropleth_immigrants(df_combined, year=2024)
fig_immigrants.show()
print("\nTop 5 Destination Countries (2024):")
for _, row in df_combined.nlargest(5, 'immigrants_2024').iterrows():
    print(f"  {row['country']}: {row['immigrants_2024']:,.0f} immigrants")

## Section 10: Choropleth Map 2 - Emigrants by Origin
Where do migrants come from? This map shows the diaspora - citizens living abroad.

In [ ]:
# =============================================================================
# SECTION 10: CHOROPLETH MAP 2 - EMIGRANTS BY ORIGIN
# =============================================================================
fig_emigrants = create_choropleth_emigrants(df_combined, year=2024)
fig_emigrants.show()
print("\nTop 5 Origin Countries (2024):")
for _, row in df_combined.nlargest(5, 'emigrants_2024').iterrows():
    print(f"  {row['country']}: {row['emigrants_2024']:,.0f} citizens abroad")

## Section 11: Choropleth Map 3 - Net Migration
Which countries gain or lose population through migration?
- **Orange**: Net gain (more immigrants than emigrants)
- **Purple**: Net loss (more emigrants than immigrants)

In [ ]:
# =============================================================================
# SECTION 11: CHOROPLETH MAP 3 - NET MIGRATION
# =============================================================================
fig_net = create_choropleth_net_migration(df_combined, year=2024)
fig_net.show()
print("\nBiggest Net Receivers (2024):")
for _, row in df_combined.nlargest(3, 'net_migration_2024').iterrows():
    print(f"  {row['country']}: +{row['net_migration_2024']:,.0f}")
print("\nBiggest Net Senders (2024):")
for _, row in df_combined.nsmallest(3, 'net_migration_2024').iterrows():
    print(f"  {row['country']}: {row['net_migration_2024']:,.0f}")

## Section 12: Interactive Map Explorer

**Interactivity**: Select different years and map types to explore how migration patterns have changed over time.

Instructions:
1. Select a year from the dropdown
2. Select a map type (Immigrants, Emigrants, or Net Migration)
3. Click **Update Map** to refresh

In [ ]:
# =============================================================================
# SECTION 12: INTERACTIVE MAP EXPLORER
# =============================================================================
# Allows users to dynamically explore different years and metrics
# Create output area for the map
map_output = widgets.Output()
# Dropdown widgets
year_dropdown = widgets.Dropdown(
    options=YEARS,
    value=2024,
    description='Year:',
    style={'description_width': '50px'}
)
map_type_dropdown = widgets.Dropdown(
    options=[
        ('Immigrants (Destination)', 'immigrants'),
        ('Emigrants (Origin)', 'emigrants'),
        ('Net Migration', 'net')
    ],
    value='immigrants',
    description='Map Type:',
    style={'description_width': '80px'}
)
# Update button
update_map_btn = widgets.Button(
    description='Update Map',
    button_style='primary',
    icon='refresh'
)
def update_map_display(b):
    """Generate and display the selected map type for the selected year."""
    with map_output:
        clear_output(wait=True)

        year = year_dropdown.value
        map_type = map_type_dropdown.value

        if map_type == 'immigrants':
            fig = create_choropleth_immigrants(df_combined, year)
        elif map_type == 'emigrants':
            fig = create_choropleth_emigrants(df_combined, year)
        else:
            fig = create_choropleth_net_migration(df_combined, year)

        fig.show()
        print(f"\nDisplaying {map_type} for {year}")
update_map_btn.on_click(update_map_display)
# Layout the controls
controls = widgets.HBox([year_dropdown, map_type_dropdown, update_map_btn])
display(widgets.VBox([
    widgets.HTML('<h4>Interactive Map Explorer</h4>'),
    controls,
    map_output
]))
# Show initial map
update_map_display(None)

## Section 13: Animated Choropleth Map

Watch migration patterns evolve from 1990 to 2024. Press the **Play** button to start the animation.

In [ ]:
# =============================================================================
# SECTION 13: ANIMATED CHOROPLETH MAP
# =============================================================================
# Shows temporal evolution of migration patterns
# Create animated map output
animated_output = widgets.Output()
metric_dropdown = widgets.Dropdown(
    options=[
        ('Immigrants (Destination)', 'immigrants'),
        ('Emigrants (Origin)', 'emigrants'),
        ('Net Migration', 'net_migration')
    ],
    value='immigrants',
    description='Metric:',
    style={'description_width': '60px'}
)
animate_btn = widgets.Button(
    description='Generate Animation',
    button_style='success',
    icon='play'
)
def show_animation(b):
    """Generate and display the animated choropleth."""
    with animated_output:
        clear_output(wait=True)
        print(f"Generating animation for {metric_dropdown.value}...")
        print("This may take a few seconds...\n")

        fig = create_animated_choropleth(df_combined, metric_dropdown.value)
        fig.show()

        print("\nAnimation ready! Press Play to watch 1990-2024 evolution.")
animate_btn.on_click(show_animation)
display(widgets.VBox([
    widgets.HTML('<h4>Animated Migration Map</h4>'),
    widgets.HBox([metric_dropdown, animate_btn]),
    animated_output
]))

## Section 14: Top Migration Corridors

The largest country-to-country migration flows. These corridors reveal:
- Economic migration (Mexico-USA, Philippines-UAE)
- Forced displacement (Syria-Türkiye)
- Historical ties (India-Pakistan partition legacy)

In [ ]:
# =============================================================================
# SECTION 14: TOP MIGRATION CORRIDORS
# =============================================================================
def get_top_corridors(df, year=2024, n=15):
    """
    Get the top N migration corridors for a given year.

    Parameters:
        df: Corridor dataframe
        year: Year to analyze
        n: Number of top corridors to return
    """
    col = f'migrants_{year}'
    top = df.nlargest(n, col)[['origin', 'destination', col]].copy()
    top['corridor'] = top['origin'] + ' → ' + top['destination']
    top['migrants'] = top[col]
    return top[['corridor', 'migrants', 'origin', 'destination']]
# Get top corridors for 2024
top_corridors = get_top_corridors(df_corridors, 2024, 15)
# Create horizontal bar chart
fig_corridors = go.Figure()
fig_corridors.add_trace(go.Bar(
    y=top_corridors['corridor'][::-1],  # Reverse for largest at top
    x=top_corridors['migrants'][::-1] / 1e6,
    orientation='h',
    marker=dict(
        color=top_corridors['migrants'][::-1],
        colorscale=COLOUR_SCALE_ORANGE,
        showscale=False
    ),
    hovertemplate='%{y}<br>%{x:.2f} million<extra></extra>'
))
fig_corridors.update_layout(
    title='<b>Top 15 Migration Corridors (2024)</b><br><sup>Origin → Destination</sup>',
    xaxis_title='Migrants (millions)',
    yaxis_title='',
    height=500,
    **PLOT_LAYOUT
)
fig_corridors.show()
print("\nLargest corridor:", top_corridors.iloc[0]['corridor'])
print(f"  {top_corridors.iloc[0]['migrants']:,.0f} migrants")

In [ ]:
# =============================================================================
# SECTION 14B: INTERACTIVE CORRIDOR EXPLORER
# =============================================================================
corridor_output = widgets.Output()
corridor_year = widgets.Dropdown(
    options=YEARS,
    value=2024,
    description='Year:'
)
corridor_n = widgets.IntSlider(
    value=15,
    min=5,
    max=25,
    step=5,
    description='Top N:'
)
corridor_btn = widgets.Button(description='Update', button_style='primary')
def update_corridors(b):
    """Redraw the corridor chart for the selected year and number of corridors."""
    with corridor_output:
        clear_output(wait=True)

        year = corridor_year.value
        n = corridor_n.value
        top = get_top_corridors(df_corridors, year, n)

        fig = go.Figure()
        fig.add_trace(go.Bar(
            y=top['corridor'][::-1],
            x=top['migrants'][::-1] / 1e6,
            orientation='h',
            marker=dict(color=top['migrants'][::-1],
                        colorscale=COLOUR_SCALE_ORANGE, showscale=False),
            hovertemplate='%{y}<br>%{x:.2f}M<extra></extra>'
        ))

        fig.update_layout(
            title=f'<b>Top {n} Migration Corridors ({year})</b>',
            xaxis_title='Migrants (millions)',
            height=max(400, n * 25),
            **PLOT_LAYOUT
        )
        fig.show()
corridor_btn.on_click(update_corridors)
display(widgets.VBox([
    widgets.HTML('<h4>Interactive Corridor Explorer</h4>'),
    widgets.HBox([corridor_year, corridor_n, corridor_btn]),
    corridor_output
]))

## Section 15: Migration Trends Over Time

How have migration patterns changed from 1990 to 2024? This section explores:
- Global trends
- Country-specific trajectories
- The impact of major events (conflicts, economic crises, COVID-19)

In [ ]:
# =============================================================================
# SECTION 15: GLOBAL MIGRATION TRENDS
# =============================================================================
years = YEARS
# Calculate global totals per year
global_immigrants = [df_combined[f'immigrants_{y}'].sum() for y in years]
global_emigrants = [df_combined[f'emigrants_{y}'].sum() for y in years]
# Create dual-axis trend chart
fig_trends = make_subplots(specs=[[{"secondary_y": True}]])
fig_trends.add_trace(
    go.Scatter(
        x=years, y=[x/1e6 for x in global_immigrants],
        name='Global Migrant Stock',
        mode='lines+markers',
        line=dict(color='#d94801', width=3),
        marker=dict(size=10),
        hovertemplate='%{x}: %{y:.0f}M<extra></extra>'
    ),
    secondary_y=False
)
# Calculate year-over-year growth
yoy_growth = [0] + [(global_immigrants[i] - global_immigrants[i-1]) /
                     global_immigrants[i-1] * 100
                     for i in range(1, len(years))]
fig_trends.add_trace(
    go.Bar(
        x=years, y=yoy_growth,
        name='Period Growth %',
        marker_color='#9e9ac8',
        opacity=0.6,
        hovertemplate='%{x}: +%{y:.1f}%<extra></extra>'
    ),
    secondary_y=True
)
fig_trends.update_layout(
    title='<b>Global Migration Trends (1990-2024)</b><br><sup>Total migrant stock and period growth rate</sup>',
    xaxis_title='Year',
    legend=dict(x=0.02, y=0.98),
    height=450,
    **PLOT_LAYOUT
)
fig_trends.update_yaxes(title_text='Migrants (millions)', secondary_y=False)
fig_trends.update_yaxes(title_text='Growth Rate (%)', secondary_y=True)
fig_trends.show()
print(f"\nGlobal migrant stock grew from {global_immigrants[0]/1e6:.0f}M to {global_immigrants[-1]/1e6:.0f}M")
print(f"Overall increase: +{(global_immigrants[-1]-global_immigrants[0])/1e6:.0f}M (+{(global_immigrants[-1]/global_immigrants[0]-1)*100:.0f}%)")

In [ ]:
# =============================================================================
# SECTION 15B: INTERACTIVE COUNTRY TRENDS
# =============================================================================
trend_output = widgets.Output()
# Multi-select for countries
country_options = sorted(df_combined['country'].unique().tolist())
country_select = widgets.SelectMultiple(
    options=country_options,
    value=['Australia', 'Brazil', 'United States of America', 'Germany'],
    description='Countries:',
    rows=8,
    style={'description_width': '80px'},
    layout={'width': '300px'}
)
metric_select = widgets.RadioButtons(
    options=[
        ('Immigrants (residing in country)', 'immigrants'),
        ('Emigrants (citizens abroad)', 'emigrants'),
        ('Net Migration', 'net_migration')
    ],
    value='immigrants',
    description='Metric:'
)
trend_btn = widgets.Button(description='Plot Trends', button_style='primary', icon='line-chart')
def plot_country_trends(b):
    """Plot the selected countries' trends when the button is clicked."""
    with trend_output:
        clear_output(wait=True)

        countries = list(country_select.value)
        metric = metric_select.value

        if len(countries) == 0:
            print("Please select at least one country")
            return

        years = YEARS

        fig = go.Figure()

        # Color palette
        colors = px.colors.qualitative.Set2

        for i, country in enumerate(countries):
            row = df_combined[df_combined['country'] == country]
            if len(row) == 0:
                continue

            values = [row[f'{metric}_{y}'].values[0] for y in years]

            fig.add_trace(go.Scatter(
                x=years,
                y=[v/1e6 for v in values],
                name=country,
                mode='lines+markers',
                line=dict(color=colors[i % len(colors)], width=2),
                marker=dict(size=8),
                hovertemplate=f'{country}<br>%{{x}}: %{{y:.2f}}M<extra></extra>'
            ))

        title_map = {
            'immigrants': 'Immigrants (Foreign-born Residents)',
            'emigrants': 'Emigrants (Citizens Living Abroad)',
            'net_migration': 'Net Migration (Immigrants - Emigrants)'
        }

        fig.update_layout(
            title=f'<b>{title_map[metric]} Over Time</b>',
            xaxis_title='Year',
            yaxis_title='Millions',
            height=450,
            legend=dict(x=1.02, y=1),
            **PLOT_LAYOUT
        )

        fig.show()

        # Summary statistics
        print("\n--- Summary (1990 vs 2024) ---")
        for country in countries:
            row = df_combined[df_combined['country'] == country]
            if len(row) > 0:
                v1990 = row[f'{metric}_1990'].values[0]
                v2024 = row[f'{metric}_2024'].values[0]
                change = (v2024 - v1990) / v1990 * 100 if v1990 > 0 else 0
                print(f"{country}: {v1990/1e6:.2f}M → {v2024/1e6:.2f}M ({change:+.0f}%)")
trend_btn.on_click(plot_country_trends)
display(widgets.VBox([
    widgets.HTML('<h4>Country Migration Trends</h4>'),
    widgets.HTML('<p>Hold Ctrl/Cmd to select multiple countries</p>'),
    widgets.HBox([country_select, metric_select]),
    trend_btn,
    trend_output
]))
# Show initial plot
plot_country_trends(None)

### Section 15C: Destination Trends and Crisis vs Economic Emigration

Line charts for the largest destinations, then for emigration from crisis-affected countries and from countries with mainly economic migration.

In [ ]:
# =============================================================================
# SECTION 14: MIGRATION TRENDS - LINE CHARTS
# =============================================================================
def create_trend_chart(df, countries, metric='immigrants'):
    """
    Create line chart showing trends for selected countries.

    Parameters:
        df: Combined migration data
        countries: List of country names to display
        metric: 'immigrants' or 'emigrants'
    """
    years = YEARS

    fig = go.Figure()

    # Define a colour palette
    colours = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00',
               '#a65628', '#f781bf', '#999999', '#66c2a4', '#fc8d62']

    for i, country in enumerate(countries):
        country_data = df[df['country'] == country]
        if len(country_data) == 0:
            continue

        values = [country_data[f'{metric}_{y}'].values[0] for y in years]

        fig.add_trace(go.Scatter(
            x=years,
            y=values,
            mode='lines+markers',
            name=country,
            line=dict(color=colours[i % len(colours)], width=2),
            marker=dict(size=8),
            hovertemplate=f'{country}<br>Year: %{{x}}<br>{metric.title()}: %{{y:,.0f}}<extra></extra>'
        ))

    title_text = 'Immigrants (foreign-born residents)' if metric == 'immigrants' else 'Emigrants (citizens abroad)'

    fig.update_layout(
        title=f'<b>Migration Trends: {title_text}</b><br><sup>Selected countries, 1990-2024</sup>',
        xaxis_title='Year',
        yaxis_title='Number of People',
        legend=dict(orientation='h', yanchor='bottom', y=-0.3, x=0.5, xanchor='center'),
        height=500,
        hovermode='x unified',
        **PLOT_LAYOUT
    )

    return fig
# Display trends for major destination countries
major_destinations = ['United States of America', 'Germany', 'Saudi Arabia',
                      'United Kingdom', 'Australia', 'Canada']
fig_dest_trends = create_trend_chart(df_combined, major_destinations, 'immigrants')
fig_dest_trends.show()
for c in major_destinations:
    r = df_combined[df_combined['country'] == c].iloc[0]
    print(f"  {c}: {r['immigrants_1990']/1e6:.1f}M (1990) -> {r['immigrants_2015']/1e6:.1f}M (2015) -> {r['immigrants_2024']/1e6:.1f}M (2024)")

In [ ]:
def print_emigrant_table(countries):
    """Print emigrant stock (millions) for the given origin countries at selected years."""
    print(f"{'Country':<26}" + "".join(f"{y:>9}" for y in (1990, 2010, 2015, 2020, 2024)))
    for c in countries:
        r = df_combined[df_combined['country'] == c]
        if len(r):
            r = r.iloc[0]
            print(f"{c:<26}" + "".join(f"{r[f'emigrants_{y}']/1e6:>8.2f}M" for y in (1990, 2010, 2015, 2020, 2024)))

# =============================================================================
# SECTION 14B: EMIGRANT TRENDS - CRISIS COUNTRIES
# =============================================================================
# Compare crisis-driven vs economic emigration over time
crisis_origins = ['Syrian Arab Republic', 'Ukraine', 'Venezuela (Bolivarian Republic of)', 'Afghanistan', 'South Sudan']
economic_origins = ['India', 'Mexico', 'China', 'Philippines']
# Crisis countries
fig_crisis = create_trend_chart(df_combined, crisis_origins, 'emigrants')
fig_crisis.update_layout(
    title='<b>Crisis-Driven Emigration Trends</b><br><sup>Countries affected by conflict or instability</sup>'
)
fig_crisis.show()
print("\nEmigrants by year (UN data has 5-year steps, so exact start dates cannot be read from it):")
print_emigrant_table(crisis_origins)

In [ ]:
# =============================================================================
# SECTION 14C: ECONOMIC MIGRATION TRENDS
# =============================================================================
fig_economic = create_trend_chart(df_combined, economic_origins, 'emigrants')
fig_economic.update_layout(
    title='<b>Economic Migration Trends</b><br><sup>Countries with primarily economic migration drivers</sup>'
)
fig_economic.show()
print("\nEmigrants by year:")
print_emigrant_table(economic_origins)

## Section 16: Case Study - Australia and Brazil

A personal perspective on migration: exploring the migration patterns of Australia (destination) and Brazil (origin).

**Key Questions:**
- How has Australia's immigrant population grown?
- Where do Brazilian emigrants go?
- How do these countries compare to global patterns?

In [ ]:
# =============================================================================
# SECTION 16: AUSTRALIA & BRAZIL CASE STUDY
# =============================================================================
# Extract data for Australia and Brazil
australia = df_combined[df_combined['country'] == 'Australia'].iloc[0]
brazil = df_combined[df_combined['country'] == 'Brazil'].iloc[0]
print("=" * 70)
print("CASE STUDY: AUSTRALIA AND BRAZIL")
print("=" * 70)
# Australia profile
print("\n--- AUSTRALIA (Destination Country) ---")
print(f"Immigrants 1990: {australia['immigrants_1990']:,.0f}")
print(f"Immigrants 2024: {australia['immigrants_2024']:,.0f}")
print(f"Growth: {australia['immigrant_growth_pct']:+.0f}%")
print(f"\nNet migration 2024: {australia['net_migration_2024']:+,.0f}")
print("Australia is a significant NET RECEIVER of migrants")
# Brazil profile
print("\n--- BRAZIL (Origin Country) ---")
print(f"Emigrants 1990: {brazil['emigrants_1990']:,.0f}")
print(f"Emigrants 2024: {brazil['emigrants_2024']:,.0f}")
print(f"Growth: {brazil['emigrant_growth_pct']:+.0f}%")
print(f"\nNet migration 2024: {brazil['net_migration_2024']:+,.0f}")
print("Brazil is a NET SENDER of migrants")

In [ ]:
# =============================================================================
# SECTION 16B: VISUALISE AUSTRALIA & BRAZIL
# =============================================================================
years = YEARS
# Get time series for both countries
aus_imm = [australia[f'immigrants_{y}'] for y in years]
aus_emi = [australia[f'emigrants_{y}'] for y in years]
bra_imm = [brazil[f'immigrants_{y}'] for y in years]
bra_emi = [brazil[f'emigrants_{y}'] for y in years]
# Create comparison subplot
fig = make_subplots(rows=1, cols=2,
                   subplot_titles=('Australia: Immigration Destination',
                                   'Brazil: Emigration Origin'))
# Australia
fig.add_trace(
    go.Scatter(x=years, y=[v/1e6 for v in aus_imm],
               name='Aus Immigrants', mode='lines+markers',
              line=dict(color='#d94801', width=2)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=years, y=[v/1e6 for v in aus_emi],
              name='Aus Emigrants', mode='lines+markers',
              line=dict(color='#756bb1', width=2, dash='dash')),
    row=1, col=1
)
# Brazil
fig.add_trace(
    go.Scatter(x=years, y=[v/1e6 for v in bra_emi],
              name='Bra Emigrants', mode='lines+markers',
              line=dict(color='#756bb1', width=2)),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=years, y=[v/1e6 for v in bra_imm],
              name='Bra Immigrants', mode='lines+markers',
              line=dict(color='#d94801', width=2, dash='dash')),
    row=1, col=2
)
fig.update_layout(
    title='<b>Australia vs Brazil: Migration Profiles</b>',
    height=400,
    **PLOT_LAYOUT
)
fig.update_xaxes(title_text='Year')
fig.update_yaxes(title_text='Millions', row=1, col=1)
fig.update_yaxes(title_text='Millions', row=1, col=2)
fig.show()
print(f"\nKey Insight: Australia hosts {australia['immigrants_2024']/1e6:.1f}M immigrants while {brazil['emigrants_2024']/1e6:.1f}M Brazilians live abroad")

In [ ]:
# =============================================================================
# SECTION 16C: WHERE DO BRAZILIANS GO?
# =============================================================================
# Find top destinations for Brazilian emigrants
brazil_corridors = df_corridors[df_corridors['origin'] == 'Brazil'].copy()
top_brazil_dest = brazil_corridors.nlargest(10, 'migrants_2024')[
    ['destination', 'migrants_2024', 'migrants_1990']
].copy()
# Calculate growth
top_brazil_dest['growth_pct'] = (
    (top_brazil_dest['migrants_2024'] - top_brazil_dest['migrants_1990']) /
    top_brazil_dest['migrants_1990'] * 100
).round(0)
print("--- Top Destinations for Brazilian Emigrants (2024) ---\n")
display(top_brazil_dest.rename(columns={
    'destination': 'Destination',
    'migrants_2024': 'Brazilians 2024',
    'migrants_1990': 'Brazilians 1990',
    'growth_pct': 'Growth %'
}))
# Visualise
fig_brazil = go.Figure()
fig_brazil.add_trace(go.Bar(
    y=top_brazil_dest['destination'][::-1],
    x=top_brazil_dest['migrants_2024'][::-1] / 1e3,
    orientation='h',
    marker=dict(color=top_brazil_dest['migrants_2024'][::-1],
                colorscale=COLOUR_SCALE_PURPLE, showscale=False),
    hovertemplate='%{y}<br>%{x:.0f}K Brazilians<extra></extra>'
))
fig_brazil.update_layout(
    title='<b>Where Do Brazilians Live Abroad? (2024)</b><br><sup>Top 10 destination countries</sup>',
    xaxis_title='Brazilians (thousands)',
    height=400,
    **PLOT_LAYOUT
)
fig_brazil.show()

In [ ]:
# =============================================================================
# SECTION 16D: WHO COMES TO AUSTRALIA?
# =============================================================================
# Find top origins for Australian immigrants
australia_corridors = df_corridors[df_corridors['destination'] == 'Australia'].copy()
top_aus_origin = australia_corridors.nlargest(10, 'migrants_2024')[
    ['origin', 'migrants_2024', 'migrants_1990']
].copy()
top_aus_origin['growth_pct'] = (
    (top_aus_origin['migrants_2024'] - top_aus_origin['migrants_1990']) /
    top_aus_origin['migrants_1990'] * 100
).round(0)
print("--- Top Origins for Immigrants to Australia (2024) ---\n")
display(top_aus_origin.rename(columns={
    'origin': 'Origin Country',
    'migrants_2024': 'In Australia 2024',
    'migrants_1990': 'In Australia 1990',
    'growth_pct': 'Growth %'
}))
# Visualise
fig_aus = go.Figure()
fig_aus.add_trace(go.Bar(
    y=top_aus_origin['origin'][::-1],
    x=top_aus_origin['migrants_2024'][::-1] / 1e3,
    orientation='h',
    marker=dict(color=top_aus_origin['migrants_2024'][::-1],
                colorscale=COLOUR_SCALE_ORANGE, showscale=False),
    hovertemplate='%{y}<br>%{x:.0f}K in Australia<extra></extra>'
))
fig_aus.update_layout(
    title='<b>Where Do Australian Immigrants Come From? (2024)</b><br><sup>Top 10 origin countries</sup>',
    xaxis_title='Immigrants (thousands)',
    height=400,
    **PLOT_LAYOUT
)
fig_aus.show()
# Check if Brazil is in top origins
brazil_to_aus = australia_corridors[australia_corridors['origin'] == 'Brazil']
if len(brazil_to_aus) > 0:
    bra_aus_2024 = brazil_to_aus['migrants_2024'].values[0]
    bra_aus_1990 = brazil_to_aus['migrants_1990'].values[0]
    print(f"\nBrazilians in Australia: {bra_aus_2024:,.0f} (up from {bra_aus_1990:,.0f} in 1990)")

### Section 16E: Australia in More Detail

The 15 largest origin countries of Australia's immigrants, and Australia's foreign-born population and net migration over time.

In [ ]:
# =============================================================================
# SECTION 18B: AUSTRALIA IMMIGRATION VISUALISATION
# =============================================================================
# Top origins of immigrants in Australia
aus_origins = df_corridors[df_corridors['destination'] == 'Australia'].copy()
aus_origins = aus_origins.nlargest(15, 'migrants_2024')
fig_aus = go.Figure()
fig_aus.add_trace(go.Bar(
    y=aus_origins['origin'],
    x=aus_origins['migrants_2024'],
    orientation='h',
    marker_color='#fd8d3c',
    hovertemplate='%{y}<br>%{x:,.0f} immigrants<extra></extra>'
))
fig_aus.update_layout(
    title='<b>Top 15 Countries of Origin for Immigrants in Australia (2024)</b>',
    xaxis_title='Number of Immigrants',
    yaxis_title='',
    height=500,
    yaxis={'categoryorder': 'total ascending'},
    **PLOT_LAYOUT
)
fig_aus.show()
print("\nAustralia's diverse immigrant population reflects its points-based immigration system")
print("and geographic position in the Asia-Pacific region.")

In [ ]:
# =============================================================================
# SECTION 18C: AUSTRALIA MIGRATION TRENDS OVER TIME
# =============================================================================
# Create detailed Australia trend
years = YEARS
aus_imm = [australia[f'immigrants_{y}'] for y in years]
aus_net = [df_combined[df_combined['country'] == 'Australia'][f'net_migration_{y}'].values[0] for y in years]
fig_aus_trend = make_subplots(rows=1, cols=2,
                              subplot_titles=('Foreign-born Population', 'Net Migration'))
fig_aus_trend.add_trace(
    go.Scatter(
        x=years, y=[v/1e6 for v in aus_imm],
        mode='lines+markers+text',
        text=[f'{v/1e6:.1f}M' for v in aus_imm],
        textposition='top center',
        line=dict(color='#d94801', width=3),
        marker=dict(size=10)
    ),
    row=1, col=1
)
fig_aus_trend.add_trace(
    go.Scatter(
        x=years, y=[v/1e6 for v in aus_net],
        mode='lines+markers+text',
        text=[f'+{v/1e6:.1f}M' for v in aus_net],
        textposition='top center',
        line=dict(color='#2ca02c', width=3),
        marker=dict(size=10)
    ),
    row=1, col=2
)
fig_aus_trend.update_layout(
    title='<b>Australia: Migration Trends (1990-2024)</b>',
    showlegend=False,
    height=400,
    **PLOT_LAYOUT
)
fig_aus_trend.update_yaxes(title_text='Millions', row=1, col=1)
fig_aus_trend.update_yaxes(title_text='Millions', row=1, col=2)
fig_aus_trend.show()
print("\nAustralia consistently shows positive net migration, making it one of")
print("the world's most significant immigrant-receiving nations per capita.")

## Section 17: Growth Rate Analysis

Which countries have experienced the most dramatic changes in migration?

This scatter plot compares:
- **X-axis**: Immigrant growth (foreign-born population change)
- **Y-axis**: Emigrant growth (diaspora change)
- **Size**: Total 2024 migrant stock

Look for outliers that indicate major migration events (conflicts, economic crises).

In [ ]:
# =============================================================================
# SECTION 17: GROWTH RATE SCATTER PLOT
# =============================================================================
# Filter out extreme outliers for better visualisation
df_growth = df_combined[
    (df_combined['immigrant_growth_pct'].between(-100, 2000)) &
    (df_combined['emigrant_growth_pct'].between(-100, 5000))
].copy()
# Calculate total migrants for bubble size
df_growth['total_2024'] = df_growth['immigrants_2024'] + df_growth['emigrants_2024']
# Identify crisis countries for highlighting
crisis_countries = ['Syrian Arab Republic', 'Ukraine', 'Venezuela (Bolivarian Republic of)', 'Afghanistan', 'South Sudan']
df_growth['category'] = df_growth['country'].apply(
    lambda x: 'Crisis-driven' if x in crisis_countries else 'Other')
# Create scatter plot
fig_growth = px.scatter(
    df_growth,
    x='immigrant_growth_pct',
    y='emigrant_growth_pct',
    size='total_2024',
    color='category',
    hover_name='country',
    hover_data={
        'immigrant_growth_pct': ':.0f',
        'emigrant_growth_pct': ':.0f',
        'total_2024': ':,.0f',
        'category': False
    },
    color_discrete_map={'Crisis-driven': '#d62728', 'Other': '#1f77b4'},
    size_max=50,
    labels={
        'immigrant_growth_pct': 'Immigrant Growth % (1990-2024)',
        'emigrant_growth_pct': 'Emigrant Growth % (1990-2024)'
    }
)
# Add reference lines
fig_growth.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig_growth.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.5)
# Add global average lines
avg_imm = df_growth['immigrant_growth_pct'].median()
avg_emi = df_growth['emigrant_growth_pct'].median()
fig_growth.add_hline(y=avg_emi, line_dash='dot', line_color='green',
                     annotation_text=f'Median emigrant growth: {avg_emi:.0f}%')
fig_growth.add_vline(x=avg_imm, line_dash='dot', line_color='green',
                    annotation_text=f'Median immigrant growth: {avg_imm:.0f}%')
fig_growth.update_layout(
    title='<b>Migration Growth Rates by Country (1990-2024)</b><br><sup>Bubble size = total 2024 migration stock; Red = crisis countries</sup>',
    height=550,
    **PLOT_LAYOUT
)
fig_growth.show()
print("\nCrisis countries (red) show extreme emigrant growth due to conflict/persecution.")
print("Countries in upper-right quadrant are both receiving AND sending more migrants.")

### Section 17B: Growth Against Size

The same growth rates against the absolute 2024 stock, for immigrants and then for emigrants.

In [ ]:
# =============================================================================
# SECTION 16: GROWTH RATE SCATTER PLOT
# =============================================================================
def create_growth_scatter(df, metric='immigrant', top_n=50):
    """
    Scatter plot showing growth rates vs absolute numbers.
    Helps identify countries with dramatic changes.

    Parameters:
        df: Combined migration data
        metric: 'immigrant' or 'emigrant'
        top_n: Number of countries to show
    """
    # Get relevant columns
    if metric == 'immigrant':
        value_col = 'immigrants_2024'
        growth_col = 'immigrant_growth_pct'
        title = 'Immigrant Growth Analysis'
        subtitle = 'Foreign-born population change 1990-2024'
    else:
        value_col = 'emigrants_2024'
        growth_col = 'emigrant_growth_pct'
        title = 'Emigrant Growth Analysis'
        subtitle = 'Citizens abroad change 1990-2024'

    # Get top countries by 2024 value
    plot_df = df.nlargest(top_n, value_col).copy()

    # Cap growth for visualisation (some are extreme)
    plot_df['growth_capped'] = plot_df[growth_col].clip(-100, 2000)

    # Size based on absolute numbers
    plot_df['size'] = np.sqrt(plot_df[value_col]) / 100

    fig = px.scatter(
        plot_df,
        x='growth_capped',
        y=value_col,
        size='size',
        hover_name='country',
        hover_data={
            value_col: ':,.0f',
            growth_col: ':+.0f',
            'growth_capped': False,
            'size': False
        },
        color='growth_capped',
        color_continuous_scale=COLOUR_SCALE_DIVERGING,
        labels={
            value_col: f'{metric.title()}s (2024)',
            'growth_capped': 'Growth % (1990-2024)'
        },
        title=f'<b>{title}</b><br><sup>{subtitle}</sup>'
    )

    # Add vertical line at 0% growth
    fig.add_vline(x=0, line_dash='dash', line_color='gray')

    fig.update_layout(
        height=550,
        coloraxis_colorbar=dict(title='Growth %'),
        **PLOT_LAYOUT
    )

    return fig
# Immigrant growth
fig_imm_growth = create_growth_scatter(df_combined, 'immigrant', top_n=50)
fig_imm_growth.show()
print("\nBubble size = population, colour = growth rate")
print("Countries on the right experienced explosive growth")

In [ ]:
# =============================================================================
# SECTION 16B: EMIGRANT GROWTH SCATTER
# =============================================================================
fig_emi_growth = create_growth_scatter(df_combined, 'emigrant', top_n=50)
fig_emi_growth.show()
print("\nSyria, Venezuela and Ukraine appear among the highest growth rates, which is consistent with crisis-driven displacement.")

## Section 18: Interactive Country Comparison Tool

Compare any two countries side-by-side across all migration metrics.

In [ ]:
# =============================================================================
# SECTION 18: INTERACTIVE COUNTRY COMPARISON
# =============================================================================
comparison_output = widgets.Output()
country1_select = widgets.Dropdown(
    options=sorted(df_combined['country'].unique().tolist()),
    value='Australia',
    description='Country 1:',
    style={'description_width': '80px'}
)
country2_select = widgets.Dropdown(
    options=sorted(df_combined['country'].unique().tolist()),
    value='Brazil',
    description='Country 2:',
    style={'description_width': '80px'}
)
compare_btn = widgets.Button(description='Compare', button_style='success', icon='balance-scale')
def compare_countries(b):
    """Print and plot a side-by-side comparison of the two selected countries."""
    with comparison_output:
        clear_output(wait=True)

        c1 = df_combined[df_combined['country'] == country1_select.value].iloc[0]
        c2 = df_combined[df_combined['country'] == country2_select.value].iloc[0]

        years = YEARS

        # Create subplot with 2 rows, 2 cols
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Immigrants Over Time',
                'Emigrants Over Time',
                'Net Migration Over Time',
                'Comparison Summary (2024)'
            ),
            specs=[[{}, {}], [{}, {'type': 'bar'}]]
        )

        # Row 1, Col 1: Immigrants
        fig.add_trace(go.Scatter(
            x=years, y=[c1[f'immigrants_{y}']/1e6 for y in years],
            name=country1_select.value, mode='lines+markers',
            line=dict(color='#d94801')
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=years, y=[c2[f'immigrants_{y}']/1e6 for y in years],
            name=country2_select.value, mode='lines+markers',
            line=dict(color='#756bb1')
        ), row=1, col=1)

        # Row 1, Col 2: Emigrants
        fig.add_trace(go.Scatter(
            x=years, y=[c1[f'emigrants_{y}']/1e6 for y in years],
            name=country1_select.value, mode='lines+markers',
            line=dict(color='#d94801'), showlegend=False
        ), row=1, col=2)
        fig.add_trace(go.Scatter(
            x=years, y=[c2[f'emigrants_{y}']/1e6 for y in years],
            name=country2_select.value, mode='lines+markers',
            line=dict(color='#756bb1'), showlegend=False
        ), row=1, col=2)

        # Row 2, Col 1: Net Migration
        fig.add_trace(go.Scatter(
            x=years, y=[c1[f'net_migration_{y}']/1e6 for y in years],
            name=country1_select.value, mode='lines+markers',
            line=dict(color='#d94801'), showlegend=False
        ), row=2, col=1)
        fig.add_trace(go.Scatter(
            x=years, y=[c2[f'net_migration_{y}']/1e6 for y in years],
            name=country2_select.value, mode='lines+markers',
            line=dict(color='#756bb1'), showlegend=False
        ), row=2, col=1)

        # Row 2, Col 2: Summary bar chart
        metrics = ['Immigrants 2024', 'Emigrants 2024', 'Net Migration']
        c1_vals = [c1['immigrants_2024']/1e6, c1['emigrants_2024']/1e6, c1['net_migration_2024']/1e6]
        c2_vals = [c2['immigrants_2024']/1e6, c2['emigrants_2024']/1e6, c2['net_migration_2024']/1e6]

        fig.add_trace(go.Bar(
            x=metrics, y=c1_vals, name=country1_select.value,
            marker_color='#d94801', showlegend=False
        ), row=2, col=2)
        fig.add_trace(go.Bar(
            x=metrics, y=c2_vals, name=country2_select.value,
            marker_color='#756bb1', showlegend=False
        ), row=2, col=2)

        fig.update_layout(
            title=f'<b>{country1_select.value} vs {country2_select.value}</b>',
            height=600,
            barmode='group',
            **PLOT_LAYOUT
        )

        fig.update_yaxes(title_text='Millions', row=1, col=1)
        fig.update_yaxes(title_text='Millions', row=1, col=2)
        fig.update_yaxes(title_text='Millions', row=2, col=1)
        fig.update_yaxes(title_text='Millions', row=2, col=2)

        fig.show()

        # Summary table
        print(f"\n{'='*60}")
        print(f"COMPARISON SUMMARY")
        print(f"{'='*60}")
        print(f"{'Metric':<25} {country1_select.value:>15} {country2_select.value:>15}")
        print(f"{'-'*60}")
        print(f"{'Immigrants 2024':<25} {c1['immigrants_2024']:>15,.0f} {c2['immigrants_2024']:>15,.0f}")
        print(f"{'Emigrants 2024':<25} {c1['emigrants_2024']:>15,.0f} {c2['emigrants_2024']:>15,.0f}")
        print(f"{'Net Migration 2024':<25} {c1['net_migration_2024']:>+15,.0f} {c2['net_migration_2024']:>+15,.0f}")
        print(f"{'Immigrant Growth %':<25} {c1['immigrant_growth_pct']:>+15.0f}% {c2['immigrant_growth_pct']:>+15.0f}%")
        print(f"{'Emigrant Growth %':<25} {c1['emigrant_growth_pct']:>+15.0f}% {c2['emigrant_growth_pct']:>+15.0f}%")
compare_btn.on_click(compare_countries)
display(widgets.VBox([
    widgets.HTML('<h4>Country Comparison Tool</h4>'),
    widgets.HBox([country1_select, country2_select, compare_btn]),
    comparison_output
]))
# Show initial comparison
compare_countries(None)

## Section 19: Post-COVID Migration Shifts (2020-2024)

Hypothesis: "After 2021 (post-COVID-19), people started to migrate more."

Let's test this with the data, examining which countries saw the biggest changes since 2020.

In [ ]:
# =============================================================================
# SECTION 19: POST-COVID MIGRATION ANALYSIS
# =============================================================================
# Calculate 2020-2024 changes
df_combined['imm_change_2020_2024'] = df_combined['immigrants_2024'] - df_combined['immigrants_2020']
df_combined['emi_change_2020_2024'] = df_combined['emigrants_2024'] - df_combined['emigrants_2020']
df_combined['imm_pct_2020_2024'] = (df_combined['imm_change_2020_2024'] /
                                    df_combined['immigrants_2020'] * 100).round(1)
df_combined['emi_pct_2020_2024'] = (df_combined['emi_change_2020_2024'] /
                                    df_combined['emigrants_2020'] * 100).round(1)
print("=" * 70)
print("POST-COVID MIGRATION CHANGES (2020-2024)")
print("=" * 70)
# Global change
global_2020 = df_combined['immigrants_2020'].sum()
global_2024 = df_combined['immigrants_2024'].sum()
global_change = global_2024 - global_2020
global_pct = (global_change / global_2020 * 100)
print(f"\nGlobal migrant stock:")
print(f"  2020: {global_2020/1e6:.0f} million")
print(f"  2024: {global_2024/1e6:.0f} million")
print(f"  Change: +{global_change/1e6:.0f} million (+{global_pct:.1f}%)")
# Countries with biggest absolute increases in immigrants
print("\n--- Biggest Immigration Increases (2020-2024) ---")
top_imm_increase = df_combined.nlargest(10, 'imm_change_2020_2024')[
    ['country', 'imm_change_2020_2024', 'imm_pct_2020_2024']]
for _, row in top_imm_increase.iterrows():
    print(f"  {row['country']}: +{row['imm_change_2020_2024']:,.0f} ({row['imm_pct_2020_2024']:+.0f}%)")
# Countries with biggest emigration surges (crisis indicators)
print("\n--- Biggest Emigration Surges (2020-2024) ---")
top_emi_increase = df_combined.nlargest(10, 'emi_change_2020_2024')[
    ['country', 'emi_change_2020_2024', 'emi_pct_2020_2024']]
for _, row in top_emi_increase.iterrows():
    print(f"  {row['country']}: +{row['emi_change_2020_2024']:,.0f} ({row['emi_pct_2020_2024']:+.0f}%)")

In [ ]:
# =============================================================================
# SECTION 19B: VISUALISE POST-COVID CHANGES
# =============================================================================
# Create butterfly chart showing top gainers and losers
fig_covid = make_subplots(rows=1, cols=2,
                          subplot_titles=('Top Immigration Increases',
                                         'Top Emigration Increases'))
# Top immigration increases
top_imm = df_combined.nlargest(10, 'imm_change_2020_2024')
fig_covid.add_trace(go.Bar(
    y=top_imm['country'][::-1],
    x=top_imm['imm_change_2020_2024'][::-1] / 1e6,
    orientation='h',
    marker_color='#d94801',
    hovertemplate='%{y}<br>+%{x:.2f}M<extra></extra>'
), row=1, col=1)
# Top emigration increases
top_emi = df_combined.nlargest(10, 'emi_change_2020_2024')
fig_covid.add_trace(go.Bar(
    y=top_emi['country'][::-1],
    x=top_emi['emi_change_2020_2024'][::-1] / 1e6,
    orientation='h',
    marker_color='#756bb1',
    hovertemplate='%{y}<br>+%{x:.2f}M<extra></extra>'
), row=1, col=2)
fig_covid.update_layout(
    title='<b>Post-COVID Migration Shifts (2020-2024)</b><br><sup>Absolute change in migrant populations</sup>',
    showlegend=False,
    height=450,
    **PLOT_LAYOUT
)
fig_covid.update_xaxes(title_text='Change (millions)', row=1, col=1)
fig_covid.update_xaxes(title_text='Change (millions)', row=1, col=2)
fig_covid.show()
print("\nConclusion: the data is consistent with the hypothesis. The global migrant stock grew faster per year in 2020-2024 than in 2015-2020.")
print("This is stock data, not yearly flows, and the comparison is not a causal test.")
print("Key drivers include: Ukraine conflict, continued displacement from Syria/Venezuela/Afghanistan,")
print("and post-pandemic economic recovery encouraging labour migration.")

### Section 19C: Crisis Timeline and Regional Growth

Emigration from four crisis-affected countries, then immigration growth in the major destination regions.

In [ ]:
# =============================================================================
# SECTION 19C: CRISIS-DRIVEN EMIGRATION TIMELINE
# =============================================================================
# Create timeline showing crisis countries
crisis_timeline = df_combined[df_combined['country'].isin(
    ['Ukraine', 'Syrian Arab Republic', 'Venezuela (Bolivarian Republic of)', 'Afghanistan'])]
years = YEARS
fig_crisis_timeline = go.Figure()
colours = {'Ukraine': '#0057B7', 'Syrian Arab Republic': '#E41A1C',
           'Venezuela (Bolivarian Republic of)': '#984EA3', 'Afghanistan': '#4DAF4A'}
for _, row in crisis_timeline.iterrows():
    country = row['country']
    values = [row[f'emigrants_{y}'] for y in years]

    fig_crisis_timeline.add_trace(go.Scatter(
        x=years, y=[v/1e6 for v in values],
        mode='lines+markers',
        name=country,
        line=dict(color=colours.get(country, '#888888'), width=2),
        marker=dict(size=8)
    ))
# Add annotations for key events
fig_crisis_timeline.add_annotation(x=2015, y=4, text="Syria Civil War", showarrow=True, arrowhead=2)
fig_crisis_timeline.add_annotation(x=2022, y=6.3, text="Ukraine Invasion", showarrow=True, arrowhead=2)
fig_crisis_timeline.add_annotation(x=2018, y=3, text="Venezuela Crisis", showarrow=True, arrowhead=2)
fig_crisis_timeline.update_layout(
    title='<b>Crisis-Driven Emigration Timeline</b><br><sup>How conflict drives displacement</sup>',
    xaxis_title='Year',
    yaxis_title='Emigrants (Millions)',
    legend=dict(orientation='h', y=-0.2),
    height=500,
    **PLOT_LAYOUT
)
fig_crisis_timeline.show()
print("\nEmigrants by year:")
print_emigrant_table(["Ukraine", "Syrian Arab Republic", "Venezuela", "Afghanistan"])

In [ ]:
# =============================================================================
# SECTION 19: GLOBAL DEBATE DATA CONTEXT
# =============================================================================
# Compare migration growth in major destination regions
debate_countries = {
    'Europe': ['Germany', 'France', 'United Kingdom', 'Italy', 'Spain'],
    'North America': ['United States of America', 'Canada'],
    'Oceania': ['Australia', 'New Zealand']
}
print("=" * 70)
print("MIGRATION GROWTH IN MAJOR DESTINATIONS")
print("=" * 70)
region_growth = {}
for region, countries in debate_countries.items():
    subset = df_combined[df_combined['country'].isin(countries)]
    imm_1990 = subset['immigrants_1990'].sum()
    imm_2024 = subset['immigrants_2024'].sum()
    growth = (imm_2024 - imm_1990) / imm_1990 * 100

    region_growth[region] = {
        '1990': imm_1990,
        '2024': imm_2024,
        'growth': growth,
        'change': imm_2024 - imm_1990
    }

    print(f"\n{region}:")
    print(f"  1990: {imm_1990/1e6:.1f} million immigrants")
    print(f"  2024: {imm_2024/1e6:.1f} million immigrants")
    print(f"  Change: +{(imm_2024-imm_1990)/1e6:.1f} million (+{growth:.0f}%)")
# Visualise
regions = list(region_growth.keys())
growth_values = [region_growth[r]['growth'] for r in regions]
change_values = [region_growth[r]['change']/1e6 for r in regions]
fig_debate = make_subplots(rows=1, cols=2,
                           subplot_titles=('Percentage Growth', 'Absolute Change (Millions)'))
fig_debate.add_trace(
    go.Bar(x=regions, y=growth_values, marker_color='#fd8d3c',
           text=[f'+{v:.0f}%' for v in growth_values], textposition='outside'),
    row=1, col=1
)
fig_debate.add_trace(
    go.Bar(x=regions, y=change_values, marker_color='#9e9ac8',
           text=[f'+{v:.1f}M' for v in change_values], textposition='outside'),
    row=1, col=2
)
fig_debate.update_layout(
    title='<b>Immigration Growth in Major Destination Regions (1990-2024)</b>',
    showlegend=False,
    height=400,
    **PLOT_LAYOUT
)
fig_debate.show()
print("\nKey insight: All major destination regions show substantial growth,")
print("which explains why immigration has become a central political issue globally.")

### Section 19E: Single-Country Dashboard

Pick one country to see its immigrants, emigrants, net migration and growth in one panel.

In [ ]:
# =============================================================================
# SECTION 19E: SINGLE-COUNTRY DASHBOARD
# =============================================================================
def create_country_dashboard(country_name):
    """
    Create a comprehensive dashboard for a single country.
    Shows immigrants, emigrants, net migration, and key statistics.
    """
    country_data = df_combined[df_combined['country'] == country_name]

    if len(country_data) == 0:
        print(f"Country '{country_name}' not found")
        return

    row = country_data.iloc[0]
    years = YEARS

    # Create subplot dashboard
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Immigration & Emigration Over Time',
            'Net Migration Over Time',
            'Key Statistics (2024)',
            'Growth Rates'
        ),
        specs=[
            [{'type': 'scatter'}, {'type': 'scatter'}],
            [{'type': 'bar'}, {'type': 'bar'}]
        ]
    )

    # Plot 1: Immigration and Emigration trends
    imm_values = [row[f'immigrants_{y}'] for y in years]
    emi_values = [row[f'emigrants_{y}'] for y in years]

    fig.add_trace(
        go.Scatter(x=years, y=imm_values, name='Immigrants',
                   line=dict(color='#fd8d3c', width=2), mode='lines+markers'),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=years, y=emi_values, name='Emigrants',
                  line=dict(color='#9e9ac8', width=2), mode='lines+markers'),
        row=1, col=1
    )

    # Plot 2: Net migration over time
    net_values = [row[f'net_migration_{y}'] for y in years]
    colours = ['#fd8d3c' if v > 0 else '#7b3294' for v in net_values]

    fig.add_trace(
        go.Bar(x=years, y=net_values, marker_color=colours, name='Net Migration',
              showlegend=False),
        row=1, col=2
    )

    # Plot 3: Key statistics
    stats = ['Immigrants', 'Emigrants', 'Net Migration']
    values = [row['immigrants_2024'], row['emigrants_2024'], row['net_migration_2024']]

    fig.add_trace(
        go.Bar(x=stats, y=values,
               marker_color=['#fd8d3c', '#9e9ac8', '#66c2a4' if values[2] > 0 else '#7b3294'],
               showlegend=False,
               text=[f'{v:,.0f}' for v in values], textposition='outside'),
        row=2, col=1
    )

    # Plot 4: Growth rates
    growth_labels = ['1990-2024\nImmigrant', '1990-2024\nEmigrant', 'Post-COVID\nImmigrant']
    growth_values = [row['immigrant_growth_pct'], row['emigrant_growth_pct'],
                     row['imm_pct_2020_2024'] if pd.notna(row['imm_pct_2020_2024']) else 0]

    fig.add_trace(
        go.Bar(x=growth_labels, y=growth_values,
               marker_color=['#fd8d3c', '#9e9ac8', '#66c2a4'],
               showlegend=False,
               text=[f'{v:+.0f}%' for v in growth_values], textposition='outside'),
        row=2, col=2
    )

    fig.update_layout(
        title=f'<b>Migration Dashboard: {country_name}</b>',
        height=700,
        showlegend=True,
        legend=dict(orientation='h', y=1.1),
        **PLOT_LAYOUT
    )

    return fig
# Interactive country selector
dashboard_country = widgets.Dropdown(
    options=sorted(df_combined['country'].unique().tolist()),
    value='Australia',
    description='Country:',
    style={'description_width': '80px'}
)
dashboard_output = widgets.Output()
dashboard_btn = widgets.Button(description='Show Dashboard', button_style='primary')
def show_country_dashboard(b):
    """Show the single-country dashboard for the selected country."""
    with dashboard_output:
        clear_output(wait=True)
        fig = create_country_dashboard(dashboard_country.value)
        if fig:
            fig.show()
dashboard_btn.on_click(show_country_dashboard)
print("Select a country to see comprehensive migration statistics:\n")
display(widgets.HBox([dashboard_country, dashboard_btn]))
display(dashboard_output)
# Show Australia by default
show_country_dashboard(None)

## Section 20: Summary Statistics Tables

In [ ]:
# =============================================================================
# SECTION 20: SUMMARY STATISTICS TABLES
# =============================================================================
print("=" * 70)
print("SUMMARY STATISTICS")
print("=" * 70)
# Table 1: Top 10 Destinations
print("\n--- TABLE 1: Top 10 Destination Countries (2024) ---")
table1 = df_combined.nlargest(10, 'immigrants_2024')[
    ['country', 'immigrants_2024', 'immigrants_1990', 'immigrant_growth_pct']
].copy()
table1.columns = ['Country', 'Immigrants 2024', 'Immigrants 1990', 'Growth %']
table1['Immigrants 2024'] = table1['Immigrants 2024'].apply(lambda x: f'{x:,.0f}')
table1['Immigrants 1990'] = table1['Immigrants 1990'].apply(lambda x: f'{x:,.0f}')
table1['Growth %'] = table1['Growth %'].apply(lambda x: f'{x:+.0f}%')
display(table1.reset_index(drop=True))
# Table 2: Top 10 Origins
print("\n--- TABLE 2: Top 10 Origin Countries (2024) ---")
table2 = df_combined.nlargest(10, 'emigrants_2024')[
    ['country', 'emigrants_2024', 'emigrants_1990', 'emigrant_growth_pct']
].copy()
table2.columns = ['Country', 'Emigrants 2024', 'Emigrants 1990', 'Growth %']
table2['Emigrants 2024'] = table2['Emigrants 2024'].apply(lambda x: f'{x:,.0f}')
table2['Emigrants 1990'] = table2['Emigrants 1990'].apply(lambda x: f'{x:,.0f}')
table2['Growth %'] = table2['Growth %'].apply(lambda x: f'{x:+.0f}%')
display(table2.reset_index(drop=True))
# Table 3: Fastest Growing
print("\n--- TABLE 3: Fastest Growing Immigrant Populations (1990-2024) ---")
# Filter for countries with significant population (>100k in 2024)
growing = df_combined[df_combined['immigrants_2024'] > 100000]
table3 = growing.nlargest(10, 'immigrant_growth_pct')[
    ['country', 'immigrants_1990', 'immigrants_2024', 'immigrant_growth_pct']
].copy()
table3.columns = ['Country', '1990', '2024', 'Growth %']
table3['1990'] = table3['1990'].apply(lambda x: f'{x:,.0f}')
table3['2024'] = table3['2024'].apply(lambda x: f'{x:,.0f}')
table3['Growth %'] = table3['Growth %'].apply(lambda x: f'{x:+,.0f}%')
display(table3.reset_index(drop=True))

## Section 21: Choropleth Map 4 - Migration Growth Rates

This map shows which countries have experienced the fastest-growing immigrant populations since 1990.
High growth rates often indicate:
- Economic development attracting workers (Gulf states)
- Proximity to crisis zones (Türkiye, Uganda)
- Policy changes enabling immigration

In [ ]:
# =============================================================================
# SECTION 21: CHOROPLETH MAP 4 - GROWTH RATES
# =============================================================================
def create_choropleth_growth(df, metric='immigrant_growth_pct'):
    """
    Choropleth Map 4: Migration growth rates (1990-2024).
    Shows which countries experienced the fastest changes.

    Parameters:
        df: DataFrame with growth data
        metric: 'immigrant_growth_pct' or 'emigrant_growth_pct'

    Returns:
        Plotly figure object
    """
    # Cap extreme values for better visualisation
    df_plot = df.copy()
    df_plot['growth_capped'] = df_plot[metric].clip(-100, 1000)

    title_map = {
        'immigrant_growth_pct': 'Immigrant Population Growth',
        'emigrant_growth_pct': 'Emigrant Population Growth'
    }

    fig = px.choropleth(
        df_plot,
        locations='country',
        locationmode='country names',
        color='growth_capped',
        hover_name='country',
        hover_data={
            metric: ':.0f',
            'growth_capped': False,
            'country': False
        },
        color_continuous_scale='RdYlGn',  # Red-Yellow-Green for growth
        range_color=[-100, 500],
        labels={metric: 'Growth %'},
        title=f'<b>Map 4: {title_map[metric]} (1990-2024)</b><br><sup>Percentage change in foreign-born/diaspora population</sup>'
    )

    fig.update_layout(**PLOT_LAYOUT, geo=GEO_LAYOUT,
                      coloraxis_colorbar=dict(title='Growth %', ticksuffix='%'))
    return fig
# Display growth rate choropleth
fig_growth_map = create_choropleth_growth(df_combined, 'immigrant_growth_pct')
fig_growth_map.show()
print("\nGreen = High growth | Yellow = Moderate | Red = Decline/Low growth")
print("Republic of Korea, Peru and Colombia show some of the highest immigrant growth rates, and Gulf states such as Qatar and the UAE also grew sharply.")

In [ ]:
# =============================================================================
# SECTION 21B: INTERACTIVE GROWTH MAP
# =============================================================================
growth_map_output = widgets.Output()
growth_metric_dropdown = widgets.Dropdown(
    options=[
        ('Immigrant Growth Rate', 'immigrant_growth_pct'),
        ('Emigrant Growth Rate', 'emigrant_growth_pct')
    ],
    value='immigrant_growth_pct',
    description='Metric:'
)
growth_map_btn = widgets.Button(description='Update Map', button_style='primary')
def update_growth_map(b):
    """Redraw the growth map for the selected metric."""
    with growth_map_output:
        clear_output(wait=True)
        metric = growth_metric_dropdown.value
        fig = create_choropleth_growth(df_combined, metric)
        fig.show()
growth_map_btn.on_click(update_growth_map)
display(widgets.VBox([
    widgets.HTML('<h4>Growth Rate Map Explorer</h4>'),
    widgets.HBox([growth_metric_dropdown, growth_map_btn]),
    growth_map_output
]))

## Section 22: Regional Migration Patterns

How do different world regions compare in terms of migration?

The regions below use a hand-picked list of major countries per region (the `Countries` column shows how many were found), so totals are for those countries only, not for the whole region.


In [ ]:
# =============================================================================
# SECTION 22: REGIONAL ANALYSIS
# =============================================================================
# Selected major countries per region (not every country: the UN aggregates are excluded on purpose)
region_mapping = {
    'Europe': ['Germany', 'United Kingdom', 'France', 'Spain', 'Italy', 'Russian Federation',
               'Ukraine', 'Poland', 'Netherlands', 'Switzerland', 'Sweden', 'Austria', 'Belgium'],
    'North America': ['United States of America', 'Canada', 'Mexico'],
    'Asia-Pacific': ['Australia', 'China', 'India', 'Japan', 'Republic of Korea',
                     'Indonesia', 'Philippines', 'Thailand', 'Viet Nam', 'Malaysia', 'New Zealand'],
    'Middle East & Gulf': ['Saudi Arabia', 'United Arab Emirates', 'Türkiye', 'Israel',
                           'Jordan', 'Kuwait', 'Qatar', 'Oman', 'Bahrain'],
    'South America': ['Brazil', 'Argentina', 'Colombia', 'Chile', 'Peru', 'Venezuela (Bolivarian Republic of)'],
    'Africa': ['South Africa', 'Egypt', 'Nigeria', 'Ethiopia', 'Kenya', 'Morocco']
}
# Calculate regional totals
regional_stats = []
for region, countries in region_mapping.items():
    region_data = df_combined[df_combined['country'].isin(countries)]

    regional_stats.append({
        'Region': region,
        'Countries': len(region_data),
        'Immigrants 2024': region_data['immigrants_2024'].sum(),
        'Emigrants 2024': region_data['emigrants_2024'].sum(),
        'Net Migration 2024': region_data['net_migration_2024'].sum()
    })
df_regional = pd.DataFrame(regional_stats)
# Display
print("=" * 70)
print("REGIONAL MIGRATION SUMMARY (2024)")
print("=" * 70)
display(df_regional)
# Visualise
fig_region = go.Figure()
fig_region.add_trace(go.Bar(
    name='Immigrants',
    x=df_regional['Region'],
    y=df_regional['Immigrants 2024'] / 1e6,
    marker_color='#d94801'
))
fig_region.add_trace(go.Bar(
    name='Emigrants',
    x=df_regional['Region'],
    y=df_regional['Emigrants 2024'] / 1e6,
    marker_color='#756bb1'
))
fig_region.update_layout(
    title='<b>Regional Migration Comparison (2024)</b><br><sup>Selected major countries in each region</sup>',
    xaxis_title='Region',
    yaxis_title='Millions',
    barmode='group',
    height=450,
    **PLOT_LAYOUT
)
fig_region.show()
print("\nNorth America and Europe are major destinations.")
print("Asia-Pacific and South America are significant origins.")
print("The Middle East/Gulf region has grown dramatically as a destination.")

# Region label per country, used by the emigration sunburst in Section 24B
country_to_region = {c: r for r, cs in region_mapping.items() for c in cs}
df_combined['region'] = df_combined['country'].map(country_to_region)


## Section 23: Treemap Visualisation

An alternative way to visualise the concentration of migrants across countries.

In [ ]:
# =============================================================================
# SECTION 23: TREEMAP VISUALISATION
# =============================================================================
# Top 20 destinations for treemap
top20_dest = df_combined.nlargest(20, 'immigrants_2024').copy()
fig_tree = px.treemap(
    top20_dest,
    path=[px.Constant('World'), 'country'],
    values='immigrants_2024',
    color='immigrant_growth_pct',
    color_continuous_scale='RdYlGn',
    range_color=[-50, 300],
    hover_data={'immigrants_2024': ':,.0f', 'immigrant_growth_pct': ':.0f'},
    title='<b>Top 20 Immigrant Destinations</b><br><sup>Size = total immigrants; Colour = growth rate since 1990</sup>'
)
fig_tree.update_layout(height=500, **PLOT_LAYOUT)
fig_tree.show()
print("\nUSA dominates, but Germany, Saudi Arabia, and UAE are major destinations too.")

## Section 24: Sunburst Chart - Top Corridors

In [ ]:
# =============================================================================
# SECTION 24: SUNBURST CHART
# =============================================================================
# Get top corridors and create hierarchy
top_corr = get_top_corridors(df_corridors, 2024, 20)
fig_sun = px.sunburst(
    top_corr,
    path=['origin', 'destination'],
    values='migrants',
    color='migrants',
    color_continuous_scale=COLOUR_SCALE_ORANGE,
    title='<b>Top 20 Migration Corridors</b><br><sup>Inner ring: Origin countries, Outer ring: Destinations</sup>'
)
fig_sun.update_layout(height=550, **PLOT_LAYOUT)
fig_sun.show()
print("\nThis visualisation shows which origin countries send migrants to which destinations.")

### Section 24B: Sunburst of Emigration and Country Profiles

The sunburst shows emigration by region and country. The parallel coordinates compare 15 countries across five measures.

In [ ]:
# =============================================================================
# SECTION 20B: SUNBURST CHART - EMIGRATION PATTERNS
# =============================================================================
# Sunburst shows hierarchical relationships in circular form
emigrant_data = df_combined[df_combined['region'].notna()].copy()
emigrant_data = emigrant_data.nlargest(40, 'emigrants_2024')
fig_sunburst = px.sunburst(
    emigrant_data,
    path=['region', 'country'],
    values='emigrants_2024',
    color='emigrant_growth_pct',
    color_continuous_scale=COLOUR_SCALE_DIVERGING,
    color_continuous_midpoint=100,
    title='<b>Emigration from Selected Major Countries</b><br><sup>Click to explore regions</sup>'
)
fig_sunburst.update_layout(
    height=600,
    **PLOT_LAYOUT
)
fig_sunburst.show()
print("Click on segments to zoom in; click center to zoom out")

In [ ]:
# =============================================================================
# SECTION 20C: PARALLEL COORDINATES - COUNTRY PROFILES
# =============================================================================
# Shows relationships between multiple variables
# Select interesting countries
profile_countries = [
    'United States of America', 'Germany', 'Saudi Arabia', 'Australia', 'Canada',
    'India', 'Mexico', 'China', 'Syrian Arab Republic', 'Ukraine', 'Brazil',
    'United Kingdom', 'France', 'United Arab Emirates', 'Russian Federation'
]
profile_df = df_combined[df_combined['country'].isin(profile_countries)].copy()
# Create parallel coordinates
fig_parallel = px.parallel_coordinates(
    profile_df,
    dimensions=['immigrants_2024', 'emigrants_2024', 'net_migration_2024',
                'immigrant_growth_pct', 'imm_pct_2020_2024'],
    color='net_migration_2024',
    color_continuous_scale=COLOUR_SCALE_DIVERGING,
    labels={
        'immigrants_2024': 'Immigrants 2024',
        'emigrants_2024': 'Emigrants 2024',
        'net_migration_2024': 'Net Migration',
        'immigrant_growth_pct': 'Growth 1990-2024 %',
        'imm_pct_2020_2024': 'Growth 2020-2024 %'
    },
    title='<b>Country Migration Profiles</b><br><sup>Drag on axes to filter</sup>'
)
fig_parallel.update_layout(
    height=500,
    **PLOT_LAYOUT
)
fig_parallel.show()
print("Drag on any axis to filter countries by that dimension")
print("This reveals patterns: high-growth countries, net receivers vs senders, etc.")

## Section 25: Key Findings and Narrative

### Summary of Main Insights

In [ ]:
# =============================================================================
# SECTION 25: KEY FINDINGS (every number is computed from the data)
# =============================================================================
total = {y: df_combined[f'immigrants_{y}'].sum() for y in YEARS}
growth_pct = (total[2024] - total[1990]) / total[1990] * 100
top10 = df_combined.nlargest(10, 'immigrants_2024')
top10_share = top10['immigrants_2024'].sum() / total[2024] * 100
top_dest = top10.iloc[0]
next_four = ", ".join(top10['country'].iloc[1:5])

def country_row(name):
    """Return the combined-data row for one country."""
    return df_combined[df_combined['country'] == name].iloc[0]

def emigrant_change(name):
    """Return a country's emigrants in 1990 and 2024 and the percentage change."""
    r = country_row(name)
    return r['emigrants_1990'], r['emigrants_2024'], (r['emigrants_2024'] - r['emigrants_1990']) / r['emigrants_1990'] * 100

mex_usa = df_corridors[(df_corridors['origin'] == 'Mexico') &
                       (df_corridors['destination'] == 'United States of America')]['migrants_2024'].sum()
aus = country_row('Australia')
aus_top = df_corridors[df_corridors['destination'] == 'Australia'].nlargest(3, 'migrants_2024')['origin']
pace_before = ((total[2020] / total[2015]) ** (1 / 5) - 1) * 100   # per year, 2015-2020
pace_after = ((total[2024] / total[2020]) ** (1 / 4) - 1) * 100    # per year, 2020-2024

print("=" * 70)
print("KEY FINDINGS: THE GLOBAL MIGRATION STORY")
print("=" * 70)
print(f"""
1. SCALE OF GLOBAL MIGRATION
   - 1990: {total[1990]/1e6:.0f} million international migrants
   - 2024: {total[2024]/1e6:.0f} million international migrants
   - Growth: +{growth_pct:.0f}% over 34 years

2. CONCENTRATION OF DESTINATIONS
   - {top_dest['country']} hosts {top_dest['immigrants_2024']/1e6:.1f} million immigrants, the most of any country
   - The next largest are {next_four}
   - The 10 largest destinations host {top10_share:.0f}% of all international migrants

3. CRISIS-DRIVEN EMIGRATION""")
for name in ['Syrian Arab Republic', 'Venezuela (Bolivarian Republic of)', 'Ukraine']:
    a, b, pct = emigrant_change(name)
    print(f"   - {name}: {a/1e6:.2f}M (1990) -> {b/1e6:.2f}M (2024), {pct:+,.0f}%")
print(f"""
4. LARGEST CORRIDOR IN THE DATA
   - Mexico to the USA: {mex_usa/1e6:.1f} million migrants in 2024

5. AUSTRALIA
   - Foreign-born population: {aus['immigrants_1990']/1e6:.1f}M (1990) to {aus['immigrants_2024']/1e6:.1f}M (2024)
   - Largest origins in 2024: {', '.join(aus_top)}

6. THE POST-COVID QUESTION
   - Average growth of the global migrant stock: {pace_before:.1f}% a year in 2015-2020, {pace_after:.1f}% a year in 2020-2024
   - {'Faster' if pace_after > pace_before else 'Not faster'} after 2020 in this data. This is a comparison of two periods, not a causal test.
""")

In [ ]:
# Final summary figure
fig_summary = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Global Growth', 'Top 5 Destinations',
        'Crisis vs Economic Origins', 'Net Migration Leaders'
    )
)
# 1. Global growth
years = [1990, 2000, 2010, 2020, 2024]
totals = [total[y]/1e6 for y in years]
fig_summary.add_trace(
    go.Bar(x=[str(y) for y in years], y=totals, marker_color='#fd8d3c'),
    row=1, col=1
)
# 2. Top 5 destinations
top5 = df_combined.nlargest(5, 'immigrants_2024')
fig_summary.add_trace(
    go.Bar(y=top5['country'], x=top5['immigrants_2024']/1e6,
           orientation='h', marker_color='#9e9ac8'),
    row=1, col=2
)
# 3. Crisis vs Economic
crisis = ['Syrian Arab Republic', 'Venezuela (Bolivarian Republic of)', 'Ukraine']
economic = ['India', 'China', 'Mexico']
crisis_vals = df_combined[df_combined['country'].isin(crisis)]['emigrants_2024'].sum()/1e6
econ_vals = df_combined[df_combined['country'].isin(economic)]['emigrants_2024'].sum()/1e6
fig_summary.add_trace(
    go.Bar(x=['Crisis Origins', 'Economic Origins'], y=[crisis_vals, econ_vals],
           marker_color=['#d62728', '#2ca02c']),
    row=2, col=1
)
# 4. Net migration leaders
top_net = df_combined.nlargest(5, 'net_migration_2024')
fig_summary.add_trace(
    go.Bar(y=top_net['country'], x=top_net['net_migration_2024']/1e6,
           orientation='h', marker_color='#66c2a4'),
    row=2, col=2
)
fig_summary.update_layout(
    title='<b>Global Migration Summary</b>',
    showlegend=False,
    height=600,
    **PLOT_LAYOUT
)
fig_summary.show()

## Section 26: Technical Documentation

### Dashboard Structure

| Section | Description | Interactivity |
|---------|-------------|---------------|
| 6 | Exploratory Data Analysis | Static tables and charts |
| 7 | UNHCR Refugee Integration | Crisis vs economic comparison |
| 9-11 | Three core choropleth maps | Static (year 2024) |
| 12 | Interactive Map Explorer | Year + metric selection |
| 13 | Animated Choropleth | Play button timeline |
| 14 | Top Corridors | Year + count selection |
| 15, 15B | Global and Country Trends | Multi-country selection |
| 15C | Destination, Crisis and Economic Trends | Hover + zoom |
| 16 | Australia-Brazil Case Study | Focused analysis |
| 17, 17B | Growth Rate Scatter Plots | Hover + zoom |
| 18 | Country Comparison Tool | Two-country selection |
| 19, 19C | Post-COVID Analysis, Crisis Timeline, Regional Growth | Time-specific focus |
| 19E | Single-Country Dashboard | Country selection |
| 21 | Growth Rate Choropleth | Map 4 with interactivity |
| 22 | Regional Analysis | Grouped bar chart |
| 23 | Treemap | Hierarchical view |
| 24, 24B | Sunbursts and Parallel Coordinates | Drill-down and axis filters |
| 25 | Key Findings | Numbers computed from the data |

### Data Sources

1. **UN DESA International Migrant Stock 2024**
   - URL: https://population.un.org/
   - Coverage: 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024
   - Variables: Foreign-born population by country of residence and origin

2. **UNHCR Global Trends reports (approximate figures)**
   - The World Bank indicator SM.POP.REFG is no longer published, so five crisis origins use figures typed in from the reports
   - Used only to flag forced displacement; no headline number depends on them

### Design Principles (Tufte)

- **Data-ink ratio**: Maximised by removing unnecessary gridlines, borders
- **Chartjunk reduction**: No decorative elements, minimal legends
- **Clear typography**: Arial font, appropriate sizing, centered titles
- **Colour rationale**: Orange/Purple scales avoid blue (water) and green (vegetation) cartographic conventions
- **Interactivity**: Enables exploration without cluttering static views

---

## End of Dashboard

This dashboard explores global migration patterns from 1990 to 2024, telling the story of how and why over 300 million people have moved across borders. From economic opportunity to forced displacement, the data reveals the complex forces shaping human mobility in our interconnected world.

---